# Team pressure analysis — Barca (team 0) vs Atletico (team 1), first half

**What this notebook does.** Turns the frame-level tracking tables (`player_frame_table`, `ball_frame_table`) into pressing metrics: where and how much each team presses, what the press does to the opponent's build-up, and how each team reacts after losing the ball. It then writes two JSON files (UI + LLM).

**How to run.** Top to bottom, once. Every analysis threshold lives in cell 1.1 (export paths, the pricing table and the corner-zone diagnostic keep their own constants next to where they are used); every later cell reads from earlier stages only, never from a later one.

## Structure (dependency order)

| Part | What it builds | Main outputs |
|---|---|---|
| 1. Setup | config, data load, helpers | `player_frame_table`, `ball_frame_table` |
| 2. Preprocessing | defensive baseline, stoppages, clean outfield table, carrier lookup | `team_defensive_line_avg`, `stoppage_frame_lookup`, `outfield` |
| 3. Frame-level signals | who is pressing the carrier, which passing lanes are blocked, combined per-team flag | `off_ball_pressure`, `frame_lane_summary`, `team_pressure_full` (`is_pressing`) |
| 4. Events | pressure events, possession events, turnovers, passes, pass outcomes, receptions | `team_pressure_events`, `possession_events`, `turnovers`, `pass_table`, `outcome_events`, `reception_triggers` |
| 5. Metrics | one cell per metric family (pressing, zones, line height, PPDA, regain, counter-press, passes, outcomes, triggers, shape, timeline, **possession**), each printing one table with team names | see the metric catalogue in 6.1 |
| 6. Team profile | metric catalogue, per-team profile, headline table, narrative | `METRIC_CATALOGUE`, `team_pressing_profile`, `headline_metrics` |
| 7. Figures | defensive line pitch, pressure heatmaps | figures, `pressure_points_clean` |
| 8. Exports | UI JSON, LLM JSON, size and cost check | `pressure_analysis_export.json`, LLM files |
| Appendix | QA checkpoints, sensitivity of the pressing share, corner-zone diagnostic | printouts only |

## Definitions to know before comparing numbers

- **Out-of-possession time** (denominator of `press_time_pct` and `zone_weighted_press_pct`) counts only frames where a ball carrier is identified and the *other* team has the ball. Pressing can only be detected on those frames, so numerator and denominator use the same frames. The all-labelled-frames version is kept as `out_of_possession_seconds_labelled` for reference.
- **Stoppage** = a no-carrier gap of at least `STOPPAGE_GAP_SEC` (30 s, config 1.1). Shorter gaps (throw-ins, goal kicks, brief tracking dropouts) stay in live play. Stoppage frames are excluded from possession and timeline figures.
- **Timeline bins** are `BIN_MINUTES` long; a short trailing bin is merged into the previous one and labelled with the true end time (e.g. `40-46:53min`).
- **Two possession shares** are reported: `possession_pct_carrier` (identified-carrier frames, use this one next to the pressing metrics) and `possession_pct_labelled` (all labelled frames).

## Naming conventions (read these once)

- **`is_pressing`** — a *team-level, per-frame* flag. `True` for team X means **team X is applying pressure** to the opponent's ball carrier. Credit goes to team X. (Previously called `under_team_pressure`, which read as the opposite.)
- **`is_presser`** — a *defender-level* flag: this individual defender is pressing the carrier in this frame.
- **`under_pressure`** — a *possession-side* flag on a pass or a possession: the team **with the ball** was being pressed. Credit goes to the opponent.
- **Credit** — every metric in the catalogue states which team it describes (`pressing team` or `team in possession`).
- **Units** — `*_pct` values are 0–100. `*_sec` are seconds. `*_m` are metres. `*_m2` are square metres.
- **Zones** — always relative to the team named on the row: `own_third`, `middle_third`, `attacking_third`.
- **Team ids** — `0` = Barca (attacks toward +x, own goal at x = 0), `1` = Atletico (attacks toward -x).

## Renamed objects (old v30 name → this notebook)

| Old | New |
|---|---|
| `under_team_pressure`, `under_team_pressure_bridged` | `is_pressing`, `is_pressing_bridged` |
| `is_pressuring` (defender level) | `is_presser` |
| `pressing_intensity` (per-frame column) | `frac_defenders_pressing` |
| `tpf`, `tpf_zone`, `tpf_events`, `tpf_binned` | `team_frames`, `team_frames_zone`, `event_frames`, `live_team_frames` |
| `pft` | `frames_with_carrier` |
| `poss` | `poss_by_frame` |
| `passes_from_scratch` | `pass_table` |
| `summary` (counter-press) | `counter_press_summary` |
| `turnovers`, `turnovers_fixed`, `turnovers_annotated`, `time_to_press` | one table: `turnovers` |
| `MIN_HOLD_SEC`, `MIN_HOLD_FRAMES` | `TURNOVER_MIN_HOLD_SEC`, `TURNOVER_MIN_HOLD_FRAMES` |

The exported JSON keeps every previous key and column name, so the Streamlit UI is unaffected. Two additive keys were introduced: `metric_catalogue` (meta) and `headline_metrics` (team).

## 1. Setup

### 1.1 Config — every threshold in one place

In [ ]:
# ---- match / pitch -------------------------------------------------------------------
FPS = 25
PITCH_LENGTH = 105          # m
PITCH_WIDTH = 68            # m
TEAM_NAMES = {0: "Barca", 1: "Atletico"}

# ---- 2. preprocessing ----------------------------------------------------------------
LINE_HEIGHT_DEFENDERS_N = 3        # "back line" = average of the last N defenders
STOPPAGE_GAP_SEC = 30.0            # a no-carrier gap this long is dead time (shorter gaps: throw-ins, goal kicks, dropouts stay in live play)
STOPPAGE_GAP_FRAMES = int(STOPPAGE_GAP_SEC * FPS)
BOUNDS_TOLERANCE_M = 2.0           # positions further outside the pitch than this are homography noise
BOX_DEPTH_M = 16.5                 # own-penalty-box depth (pressure inside it is excluded)

# ---- 3.1 defender-level pressure -----------------------------------------------------
PRESSURE_DIST_THRESH_M = 5.0
CLOSING_WINDOW_FRAMES = 12         # ~0.5s @ 25fps, window used to measure closing speed
CLOSING_SPEED_THRESH_MPS = 2.5     # must be closing at least this fast to count as an active press from open play
TIGHT_MARK_DIST_M = 1.5            # inside this range, counts as pressure even without active closing
STEP_OUT_MARGIN_M = 4.0            # must be at least this far ahead of the team's own average line depth

# ---- 3.2 passing-lane pressure -------------------------------------------------------
MIN_LANE_LENGTH_M = 3.0
TIGHT_LANE_THRESH_M = 1.5
CONTESTED_LANE_THRESH_M = 3.0

# ---- 3.3 team-level pressing flag and pressure events --------------------------------
N_PRESSING_THRESH = 2              # >= this many pressers ...
FRAC_TIGHT_BLOCKED_THRESH = 0.375  # ... OR this share of passing lanes tightly blocked
MIN_EVENT_DURATION_SEC = 0.5
GAP_TOLERANCE_FRAMES = 5           # bridge gaps up to this long inside one pressure event

# ---- 4.3 turnovers and counter-press -------------------------------------------------
TURNOVER_MIN_HOLD_SEC = 1.5        # a carrier-team run must last this long to count as a real hold
TURNOVER_MIN_HOLD_FRAMES = int(TURNOVER_MIN_HOLD_SEC * FPS)
MAX_SEARCH_WINDOW_SEC = 10         # how long after a loss we look for the first press
MAX_SEARCH_WINDOW_FRAMES = MAX_SEARCH_WINDOW_SEC * FPS
COUNTER_PRESS_WINDOW_SEC = 5.0     # "fast" counter-press = first press within this long; also the regain window
COUNTER_PRESS_WINDOW_FRAMES = int(COUNTER_PRESS_WINDOW_SEC * FPS)
MAX_REGAIN_SEC = 5.0               # press -> regain: ball won back within this long of the press event ending
MAX_REGAIN_FRAMES = int(MAX_REGAIN_SEC * FPS)

# ---- 4.4 passes ----------------------------------------------------------------------
PRESSURE_LOOKBACK_SEC = 1.0        # a pass is "under pressure" if pressing was sustained in this window before release
PRESSURE_LOOKBACK_FRAMES = int(PRESSURE_LOOKBACK_SEC * FPS)
MIN_SUSTAINED_PRESSURE_FRAMES = 8  # ~0.3s @ 25fps; consecutive frames, not a single flicker
MIN_HOLD_FRAMES_PASS = 2           # ignore single-frame carrier flicker
MIN_PASS_DISTANCE_M = 3.0          # shorter than this -> duel/tackle, not a deliberate pass
MAX_RECEPTION_GAP_FRAMES = 50      # 2s @ 25fps; longer gap -> ball likely out of play

# ---- 4.5 pass outcomes ---------------------------------------------------------------
FORWARD_PROGRESS_THRESH_M = 3.0    # net advance toward the attacking goal
BACKWARD_THRESH_M = -3.0           # net retreat away from the attacking goal
FINAL_OUTCOME_ORDER = [
    "progressed_forward", "forced_sideways", "forced_backward",
    "lost_intercepted", "lost_tackled", "lost_after_long_ball",
    "retained_after_gap", "unresolved",
]
DISRUPTIVE = ["forced_backward", "forced_sideways", "lost_intercepted", "lost_tackled", "lost_after_long_ball"]

# ---- zones ---------------------------------------------------------------------------
ZONE_ORDER = ["own_third", "middle_third", "attacking_third"]
ZONE_WEIGHTS = {"own_third": 1, "middle_third": 2, "attacking_third": 3}
# a zone label from one team's perspective -> the physically equivalent label from the opponent's perspective
ZONE_FLIP = {"own_third": "attacking_third", "middle_third": "middle_third", "attacking_third": "own_third"}

# ---- 4.6 pressing triggers -----------------------------------------------------------
PRESS_ONSET_WINDOW_SEC = 2.0
PRESS_ONSET_WINDOW_FRAMES = int(PRESS_ONSET_WINDOW_SEC * FPS)
TOUCHLINE_DIST_THRESH_M = 8.0
ISOLATED_LANES_THRESH = 2          # fewer than this many open lanes = isolated
FIRST_TOUCH_WINDOW_FRAMES = int(0.4 * FPS)
STATIC_TOUCH_SPEED_MPS = 1.5

# ---- 5. metrics ----------------------------------------------------------------------
BIN_MINUTES = 5                    # timeline bin size
MIN_LAST_BIN_FRAC = 0.5            # a trailing bin shorter than this fraction of BIN_MINUTES is merged into the previous bin
MIN_BIN_FRAMES = 500               # bins with fewer frames are masked as unreliable
BACK_N = 3                         # team shape: back line / front line sizes
FRONT_N = 3
WIDTH_IQR_MULT = 1.5
MIN_PLAYERS_FOR_SHAPE = BACK_N + FRONT_N

# ---- 8. export -----------------------------------------------------------------------
HEATMAP_BIN_M = 5.0

### 1.2 Imports, data and helpers

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import paths

player_frame_table = pd.read_parquet(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
ball_frame_table = pd.read_parquet(paths.BALL_FRAME_TABLE_CACHE_PATH)

def zone_for_team(x, team):
    """Pitch x (metres) -> the zone from `team`'s own perspective (team 0 attacks +x, team 1 attacks -x)."""
    if pd.isna(x):
        return None
    if team == 0:
        if x < 35: return "own_third"
        elif x < 70: return "middle_third"
        else: return "attacking_third"
    else:
        if x > 70: return "own_third"
        elif x > 35: return "middle_third"
        else: return "attacking_third"

def _df(df, index_name=None):
    """DataFrame/Series -> list of JSON-safe records (NaN -> null, numpy types -> native)."""
    d = df.reset_index()
    if index_name is not None:
        # the restored index is always the FIRST column after reset_index(), whatever its
        # own .name was ("index" if unnamed, or something else if the index was named --
        # e.g. team_pressing_profile's index is already named "team", not "index", so a
        # rename keyed on the literal string "index" silently did nothing for that table)
        d = d.rename(columns={d.columns[0]: index_name})
    return json.loads(d.to_json(orient="records"))

def _num(x):
    """scalar (numpy or python) -> JSON-safe float/int, NaN -> null."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    return x

# ---- readable printouts: team ids (0/1) -> team names ------------------------------------------
def _is_team_level(name):
    return name is not None and "team" in str(name)

def _to_names(values):
    vals = list(values)
    ok = len(vals) > 0 and all(
        isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, (bool, np.bool_))
        and not pd.isna(v) and float(v) in (0.0, 1.0)
        for v in vals
    )
    return [TEAM_NAMES[int(v)] for v in vals] if ok else vals

def name_teams(obj):
    """Copy of a DataFrame/Series with team ids replaced by team names in every index or column level whose name contains 'team'."""
    out = obj.copy()
    idx = out.index
    if isinstance(idx, pd.MultiIndex):
        arrays = [(_to_names(idx.get_level_values(i)) if _is_team_level(idx.names[i]) else list(idx.get_level_values(i)))
                  for i in range(idx.nlevels)]
        out.index = pd.MultiIndex.from_arrays(arrays, names=idx.names)
    elif _is_team_level(idx.name):
        out.index = pd.Index(_to_names(idx), name=idx.name)
    if isinstance(out, pd.DataFrame) and _is_team_level(out.columns.name):
        out.columns = pd.Index(_to_names(out.columns), name=out.columns.name)
    return out

def show(obj, title=None, digits=1):
    """Print a table with team names, no truncation and no dtype noise."""
    if title:
        print(title)
    if isinstance(obj, pd.Series):
        obj = obj.to_frame()
    try:
        obj = obj.round(digits)
    except TypeError:
        pass
    print(name_teams(obj).to_string())
    print()

# ---- timeline bins: a short trailing bin is merged into the previous one -------------------------
BIN_SEC = BIN_MINUTES * 60
match_end_sec = (ball_frame_table["frame_idx"].max() + 1) / FPS
n_full_bins = int(match_end_sec // BIN_SEC)
merge_last_bin = (match_end_sec - n_full_bins * BIN_SEC) < MIN_LAST_BIN_FRAC * BIN_SEC and n_full_bins > 0

def assign_time_bins(frame_idx):
    """Frame index -> bin number; a trailing bin shorter than MIN_LAST_BIN_FRAC of a full bin joins the previous bin."""
    b = (frame_idx / FPS // BIN_SEC).astype(int)
    return b.where(b < n_full_bins, n_full_bins - 1) if merge_last_bin else b

def bin_label(b, last_bin):
    """'0-5min' for full bins; the final bin is labelled with the true end time (e.g. '40-46:53min')."""
    end = match_end_sec if b == last_bin else (b + 1) * BIN_SEC
    m, sec = divmod(int(round(end)), 60)
    return f"{b * BIN_MINUTES}-{m if sec == 0 else f'{m}:{sec:02d}'}min"

print(f"player_frame_table: {len(player_frame_table):,} rows | ball_frame_table: {len(ball_frame_table):,} rows")

## 2. Preprocessing

### 2.1 Team defensive baseline
Average depth of each team's back line (last-3 defenders), computed only on frames where that team is **out of possession**. Used in 3.1 to tell "stepped out to press" from "the block was already sitting there".

In [ ]:
_poss_for_baseline = ball_frame_table[["frame_idx", "possession_team"]]

_outfield_baseline = player_frame_table[player_frame_table["role"] == "player"].dropna(subset=["pitch_x", "pitch_y"]).copy()
_outfield_baseline = _outfield_baseline.merge(_poss_for_baseline, on="frame_idx", how="left")
_outfield_baseline = _outfield_baseline[
    _outfield_baseline["possession_team"].notna() &
    (_outfield_baseline["team"] != _outfield_baseline["possession_team"])
]  # keep only frames where this team is out of possession -- their real defensive shape

_outfield_baseline["dist_from_own_goal"] = np.where(
    _outfield_baseline["team"] == 0,
    _outfield_baseline["pitch_x"],
    PITCH_LENGTH - _outfield_baseline["pitch_x"]
)
_outfield_baseline["dist_from_own_goal"] = _outfield_baseline["dist_from_own_goal"].clip(0, PITCH_LENGTH)

team_defensive_line_avg = (
    _outfield_baseline.sort_values("dist_from_own_goal")
    .groupby(["frame_idx", "team"])
    .head(LINE_HEIGHT_DEFENDERS_N)
    .groupby("team")["dist_from_own_goal"]
    .mean()
    .to_dict()
)
print("team average defensive line depth, out-of-possession only (last-3 defenders, m from own goal):", {TEAM_NAMES[int(k)]: round(v, 1) for k, v in team_defensive_line_avg.items()})

### 2.2 Stoppage detection
A no-carrier gap of at least `STOPPAGE_GAP_SEC` is flagged as dead time (injury, goal, VAR, cooling break). `stoppage_frame_lookup` is a per-frame boolean used by 4.2, 5.1 and 5.12 so dead time never counts as live out-of-possession play.

In [ ]:
carrier_missing = ball_frame_table[["frame_idx", "carrier_track_id"]].sort_values("frame_idx").reset_index(drop=True)
carrier_missing["is_missing"] = carrier_missing["carrier_track_id"].isna()
carrier_missing["gap_id"] = (carrier_missing["is_missing"] != carrier_missing["is_missing"].shift()).cumsum()

gap_runs = (
    carrier_missing[carrier_missing["is_missing"]]
    .groupby("gap_id")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
    .reset_index(drop=True)
)
gap_runs["duration_sec"] = gap_runs["n_frames"] / FPS

stoppages = gap_runs[gap_runs["duration_sec"] >= STOPPAGE_GAP_SEC].reset_index(drop=True)

# vectorized frame -> is_stoppage lookup
_stoppage_starts = stoppages["start_frame"].values
_stoppage_ends = stoppages["end_frame"].values

all_frames = ball_frame_table[["frame_idx"]].drop_duplicates().sort_values("frame_idx").reset_index(drop=True)
_frame_vals = all_frames["frame_idx"].values
_idx = np.searchsorted(_stoppage_starts, _frame_vals, side="right") - 1
_valid = _idx >= 0
_in_range = np.zeros(len(_frame_vals), dtype=bool)
_idx_valid = _idx[_valid]
_in_range[_valid] = _frame_vals[_valid] <= _stoppage_ends[_idx_valid]
all_frames["is_stoppage"] = _in_range

stoppage_frame_lookup = all_frames.set_index("frame_idx")["is_stoppage"]

print(f"{len(stoppages)} stoppages >= {STOPPAGE_GAP_SEC}s flagged, covering {stoppages['n_frames'].sum()} frames "
      f"({stoppages['duration_sec'].sum():.1f}s total dead time)")

### 2.3 Clean outfield table and possession by frame
Outfield players only, positions inside the pitch (± `BOUNDS_TOLERANCE_M`), with `dist_from_own_goal` in each team's own attacking direction. Shared by 5.3 (defensive line) and 5.11 (team shape).

In [ ]:
outfield = player_frame_table[player_frame_table["role"] == "player"].copy()
outfield = outfield.dropna(subset=["pitch_x", "pitch_y"])

in_bounds = (
    (outfield["pitch_x"] >= -BOUNDS_TOLERANCE_M) & (outfield["pitch_x"] <= PITCH_LENGTH + BOUNDS_TOLERANCE_M) &
    (outfield["pitch_y"] >= -BOUNDS_TOLERANCE_M) & (outfield["pitch_y"] <= PITCH_WIDTH + BOUNDS_TOLERANCE_M)
)
outfield = outfield[in_bounds].copy()

outfield["dist_from_own_goal"] = np.where(
    outfield["team"] == 0,
    outfield["pitch_x"],
    PITCH_LENGTH - outfield["pitch_x"]
)
outfield["dist_from_own_goal"] = outfield["dist_from_own_goal"].clip(0, PITCH_LENGTH)

poss_by_frame = ball_frame_table[["frame_idx", "possession_team"]]

### 2.4 Carrier lookup
Frames where a ball carrier is identified, with the carrier's team. `team_had_ball_between` answers "did this team have the ball at any point in [first_frame, last_frame]?" and is used by both regain metrics (press → regain in 4.1, counter-press in 4.3).

In [ ]:
carrier_lookup = (
    ball_frame_table.dropna(subset=["carrier_team"])[["frame_idx", "carrier_team"]]
    .sort_values("frame_idx")
)
carrier_frame_idx = carrier_lookup["frame_idx"].values
carrier_team_vals = carrier_lookup["carrier_team"].values

def team_had_ball_between(team_id, first_frame, last_frame):
    """True if `team_id` was the identified ball carrier in any frame of [first_frame, last_frame]."""
    lo = np.searchsorted(carrier_frame_idx, first_frame, side="left")
    hi = np.searchsorted(carrier_frame_idx, last_frame, side="right")
    return bool(np.any(carrier_team_vals[lo:hi] == team_id))

## 3. Frame-level signals

### 3.1 Defender-level proximity pressure
For every off-ball defender, distance to the ball carrier. A defender counts as a **presser** (`is_presser`) only if within `PRESSURE_DIST_THRESH_M` **and** at least one of: touch-tight (`TIGHT_MARK_DIST_M`), closing on the carrier fast enough (`CLOSING_SPEED_THRESH_MPS` over `CLOSING_WINDOW_FRAMES`), or stepped out `STEP_OUT_MARGIN_M` ahead of the team's own out-of-possession line (2.1). This separates real pressing from a compact block the attacker walked into.

In [ ]:
carrier_pos = ball_frame_table[
    ["frame_idx", "carrier_track_id", "carrier_team", "possession_team", "ball_pitch_x", "ball_pitch_y"]
]

frames_with_carrier = player_frame_table.merge(carrier_pos, on="frame_idx", how="left")

off_ball = frames_with_carrier[
    (frames_with_carrier["role"] == "player") &
    frames_with_carrier["carrier_track_id"].notna() &
    (frames_with_carrier["track_id"] != frames_with_carrier["carrier_track_id"]) &
    (frames_with_carrier["team"] != frames_with_carrier["carrier_team"])
].copy()

off_ball["dist_to_carrier"] = (
    (off_ball["pitch_x"] - off_ball["ball_pitch_x"]) ** 2 +
    (off_ball["pitch_y"] - off_ball["ball_pitch_y"]) ** 2
) ** 0.5

off_ball_pressure = off_ball.dropna(subset=["dist_to_carrier"]).copy()
off_ball_pressure = off_ball_pressure.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)

# closing speed: how fast dist_to_carrier has been shrinking over the last CLOSING_WINDOW_FRAMES for this player
off_ball_pressure["dist_to_carrier_prev"] = off_ball_pressure.groupby("track_id")["dist_to_carrier"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["frame_gap"] = off_ball_pressure["frame_idx"] - off_ball_pressure.groupby("track_id")["frame_idx"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["closing_speed_mps"] = np.where(
    (off_ball_pressure["frame_gap"] > 0) & (off_ball_pressure["frame_gap"] <= CLOSING_WINDOW_FRAMES * 2),
    (off_ball_pressure["dist_to_carrier_prev"] - off_ball_pressure["dist_to_carrier"]) / (off_ball_pressure["frame_gap"] / FPS),
    np.nan
)

# stepped ahead of the team's own habitual defensive line -> not just standing in the block
off_ball_pressure["dist_from_own_goal"] = np.where(
    off_ball_pressure["team"] == 0,
    off_ball_pressure["pitch_x"],
    PITCH_LENGTH - off_ball_pressure["pitch_x"]
)
off_ball_pressure["dist_from_own_goal"] = off_ball_pressure["dist_from_own_goal"].clip(0, PITCH_LENGTH)
off_ball_pressure["team_line_avg"] = off_ball_pressure["team"].map(team_defensive_line_avg)
off_ball_pressure["stepped_out_of_line"] = (
    off_ball_pressure["dist_from_own_goal"] >= off_ball_pressure["team_line_avg"] + STEP_OUT_MARGIN_M
)

off_ball_pressure["is_presser"] = (
    (off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M) &
    (
        (off_ball_pressure["dist_to_carrier"] < TIGHT_MARK_DIST_M) |
        (off_ball_pressure["closing_speed_mps"].fillna(0) >= CLOSING_SPEED_THRESH_MPS) |
        off_ball_pressure["stepped_out_of_line"]
    )
)

team_pressure_per_frame = (
    off_ball_pressure.groupby(["frame_idx", "team", "carrier_team", "possession_team"], as_index=False)
    .agg(
        n_pressing=("is_presser", "sum"),
        n_defenders_on_pitch=("track_id", "nunique"),
        min_dist_to_carrier=("dist_to_carrier", "min"),
    )
)
team_pressure_per_frame["frac_defenders_pressing"] = (
    team_pressure_per_frame["n_pressing"] / team_pressure_per_frame["n_defenders_on_pitch"]
)

### 3.2 Passing-lane pressure
For each teammate of the carrier, is the carrier→teammate lane blocked by an opponent? A lane is *tightly blocked* within `TIGHT_LANE_THRESH_M` and *contested* within `CONTESTED_LANE_THRESH_M`. Produces per-frame lane counts and fractions in `frame_lane_summary`.

In [ ]:
carrier_frames = frames_with_carrier[(frames_with_carrier["role"] == "player") & frames_with_carrier["carrier_track_id"].notna()].copy()

teammates = carrier_frames[
    (carrier_frames["team"] == carrier_frames["carrier_team"]) &
    (carrier_frames["track_id"] != carrier_frames["carrier_track_id"])
]
opponents = carrier_frames[carrier_frames["team"] != carrier_frames["carrier_team"]]

carrier_player_pos = (
    player_frame_table[player_frame_table["is_carrier"] == True]
    [["frame_idx", "track_id", "pitch_x", "pitch_y"]]
    .rename(columns={"track_id": "carrier_track_id", "pitch_x": "carrier_x", "pitch_y": "carrier_y"})
    .drop_duplicates(subset="frame_idx", keep="first")
)

teammate_pos = teammates[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "teammate_id", "pitch_x": "tm_x", "pitch_y": "tm_y"}
)
opponent_pos = opponents[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "opponent_id", "pitch_x": "opp_x", "pitch_y": "opp_y"}
)

lanes = teammate_pos.merge(opponent_pos, on="frame_idx", how="inner")
lanes = lanes.merge(carrier_player_pos, on="frame_idx", how="inner")
lanes = lanes.dropna(subset=["carrier_x", "carrier_y"])

Ax, Ay = lanes["carrier_x"].values, lanes["carrier_y"].values
Bx, By = lanes["tm_x"].values, lanes["tm_y"].values
Px, Py = lanes["opp_x"].values, lanes["opp_y"].values

ABx, ABy = Bx - Ax, By - Ay
APx, APy = Px - Ax, Py - Ay

seg_len_sq = ABx ** 2 + ABy ** 2
seg_len_sq = np.where(seg_len_sq == 0, 1e-9, seg_len_sq)

t = (APx * ABx + APy * ABy) / seg_len_sq
t_clipped = np.clip(t, 0.0, 1.0)

closest_x = Ax + t_clipped * ABx
closest_y = Ay + t_clipped * ABy
perp_dist = np.sqrt((Px - closest_x) ** 2 + (Py - closest_y) ** 2)

lanes["t"] = t
lanes["lane_perp_dist"] = perp_dist
lanes["lane_length_m"] = np.sqrt(seg_len_sq)

valid_lanes = lanes[
    (lanes["lane_length_m"] >= MIN_LANE_LENGTH_M) &
    (lanes["t"] >= 0.15) &
    (lanes["t"] <= 0.85)
].copy()

valid_lanes["lane_tight_block"] = valid_lanes["lane_perp_dist"] < TIGHT_LANE_THRESH_M
valid_lanes["lane_contested"] = valid_lanes["lane_perp_dist"] < CONTESTED_LANE_THRESH_M

lane_status = (
    valid_lanes.groupby(["frame_idx", "teammate_id"], as_index=False)
    .agg(lane_tight_block=("lane_tight_block", "any"), lane_contested=("lane_contested", "any"))
)

frame_lane_summary = (
    lane_status.groupby("frame_idx", as_index=False)
    .agg(
        n_lanes_total=("teammate_id", "nunique"),
        n_lanes_tight_blocked=("lane_tight_block", "sum"),
        n_lanes_contested=("lane_contested", "sum"),
    )
)
frame_lane_summary["n_lanes_open"] = frame_lane_summary["n_lanes_total"] - frame_lane_summary["n_lanes_contested"]
frame_lane_summary["frac_tight_blocked"] = frame_lane_summary["n_lanes_tight_blocked"] / frame_lane_summary["n_lanes_total"]
frame_lane_summary["frac_contested"] = frame_lane_summary["n_lanes_contested"] / frame_lane_summary["n_lanes_total"]

frame_lane_summary = frame_lane_summary.merge(
    carrier_pos[["frame_idx", "carrier_team", "possession_team"]], on="frame_idx", how="left"
)

### 3.3 Team-level pressing flag (`is_pressing`)
Per team and frame: the team is **pressing** when `n_pressing >= N_PRESSING_THRESH` **or** `frac_tight_blocked >= FRAC_TIGHT_BLOCKED_THRESH`, and the ball is not inside the pressing team's own box. Lane blocking alone can qualify a frame, so a pressure event can have few proximity pressers.

Outputs: `team_pressure_full` (per team-frame, with `is_pressing`), `team_frames` (sorted copy used for event building) and the two lookups `press_lookup` / `press_flags` used by turnovers, passes and triggers.

In [ ]:
team_pressure_full = team_pressure_per_frame.merge(
    frame_lane_summary[["frame_idx", "n_lanes_total", "n_lanes_tight_blocked",
                         "n_lanes_contested", "frac_tight_blocked", "frac_contested"]],
    on="frame_idx",
    how="left"
)

ball_pos_by_frame = ball_frame_table[["frame_idx", "ball_pitch_x", "ball_pitch_y"]].rename(
    columns={"ball_pitch_x": "press_x", "ball_pitch_y": "press_y"}
)
team_pressure_full = team_pressure_full.merge(ball_pos_by_frame, on="frame_idx", how="left")

team_frames = team_pressure_full.copy()
team_frames["frac_tight_blocked"] = team_frames["frac_tight_blocked"].fillna(0)

team_frames["in_own_box"] = np.where(
    team_frames["team"] == 0,
    team_frames["press_x"] <= BOX_DEPTH_M,
    team_frames["press_x"] >= PITCH_LENGTH - BOX_DEPTH_M
)

team_frames["is_pressing"] = (
    ((team_frames["n_pressing"] >= N_PRESSING_THRESH) | (team_frames["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)) &
    (~team_frames["in_own_box"])
)

# carry the flag back onto team_pressure_full so every downstream stage uses the same definition
team_pressure_full["is_pressing"] = team_frames["is_pressing"]

# lookups used downstream: (frame, team) -> is this team pressing in that frame
press_lookup = team_pressure_full[["frame_idx", "team", "is_pressing"]].copy()
press_flags = press_lookup.set_index(["frame_idx", "team"])["is_pressing"]

print(f"{int(team_pressure_full['is_pressing'].sum()):,} team-frames flagged is_pressing "
      f"(of {len(team_pressure_full):,} team-frames with an identified carrier)")

## 4. Events

### 4.1 Pressure events
Consecutive `is_pressing` frames per team, bridging gaps of up to `GAP_TOLERANCE_FRAMES`, kept if at least `MIN_EVENT_DURATION_SEC` long. Each event is then enriched with:

- **co-press count** — `max_n_pressing`, `mean_n_pressing_event` (lone chaser vs coordinated press),
- **regained_ball** — did the pressing team have the ball within `MAX_REGAIN_SEC` of the event ending.

In [ ]:
team_frames = team_frames.sort_values(["team", "frame_idx"]).reset_index(drop=True)

events_list = []
for team_id, grp in team_frames.groupby("team"):
    grp = grp.sort_values("frame_idx").reset_index(drop=True)

    is_press = grp["is_pressing"].values.copy()
    frames = grp["frame_idx"].values
    i, n = 0, len(is_press)
    while i < n:
        if not is_press[i]:
            j = i
            while j < n and not is_press[j]:
                j += 1
            gap_frame_span = frames[j - 1] - frames[i] + 1 if j > i else 0
            if i > 0 and j < n and gap_frame_span <= GAP_TOLERANCE_FRAMES:
                is_press[i:j] = True
            i = j
        else:
            i += 1
    grp["is_pressing_bridged"] = is_press

    grp["state_change"] = (grp["is_pressing_bridged"] != grp["is_pressing_bridged"].shift()).cumsum()
    runs = (
        grp.groupby("state_change")
        .agg(
            start_frame=("frame_idx", "first"),
            end_frame=("frame_idx", "last"),
            n_frames=("frame_idx", "count"),
            is_pressing=("is_pressing_bridged", "first"),
            possession_team=("possession_team", "first"),
            mean_n_pressing=("n_pressing", "mean"),
            mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        )
        .reset_index(drop=True)
    )
    runs["team"] = team_id
    events_list.append(runs)

team_pressure_events = pd.concat(events_list, ignore_index=True)
team_pressure_events = team_pressure_events[team_pressure_events["is_pressing"]].copy()
team_pressure_events["duration_sec"] = team_pressure_events["n_frames"] / FPS
team_pressure_events = team_pressure_events[
    team_pressure_events["duration_sec"] >= MIN_EVENT_DURATION_SEC
].reset_index(drop=True)

# ---- co-press: how many defenders press at once during each event ----
copress_records = []
for team_id, grp in team_pressure_events.groupby("team"):
    frames_t = team_frames[team_frames["team"] == team_id].sort_values("frame_idx")
    fidx = frames_t["frame_idx"].values
    npress = frames_t["n_pressing"].values
    starts = np.searchsorted(fidx, grp["start_frame"].values, side="left")
    ends = np.searchsorted(fidx, grp["end_frame"].values, side="right")
    for (idx, row), s, e in zip(grp.iterrows(), starts, ends):
        seg = npress[s:e]
        copress_records.append({
            "event_index": idx,
            "max_n_pressing": seg.max() if len(seg) else np.nan,
            "mean_n_pressing_event": seg.mean() if len(seg) else np.nan,
        })

copress_df = pd.DataFrame(copress_records).set_index("event_index")
team_pressure_events = team_pressure_events.join(copress_df)

# ---- regain: did the pressing team win the ball back within MAX_REGAIN_SEC of the event ending ----
team_pressure_events["regained_ball"] = team_pressure_events.apply(
    lambda r: team_had_ball_between(r["team"], r["end_frame"], r["end_frame"] + MAX_REGAIN_FRAMES), axis=1
)

print(f"{len(team_pressure_events)} pressure events "
      f"({{TEAM_NAMES[int(k)]: v for k, v in team_pressure_events.groupby('team').size().items()}} by team)")

### 4.2 Possession events
The match is split into possession events by forward-filling every no-carrier gap onto the last known `possession_team`, with a hard break at every stoppage. For each event, the out-of-possession team's `is_pressing` flag is averaged over the event's frames (`pct_frames_under_pressure` — the share of the possession during which the possessing team was being pressed).

In [ ]:
poss_raw = ball_frame_table[["frame_idx", "possession_team"]].sort_values("frame_idx").reset_index(drop=True)
poss_raw = poss_raw.merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
poss_raw["is_stoppage"] = poss_raw["is_stoppage"].fillna(False)

poss_raw["possession_team_filled"] = poss_raw["possession_team"].ffill()
poss_raw = poss_raw.dropna(subset=["possession_team_filled"]).reset_index(drop=True)
poss_raw["possession_team_filled"] = poss_raw["possession_team_filled"].astype(int)

# force a hard break at every stoppage -- even if the same team has the ball before and after,
# dead time should never let two possession events merge into one
poss_raw["grouping_state"] = np.where(poss_raw["is_stoppage"], -1, poss_raw["possession_team_filled"])
poss_raw["event_id"] = (poss_raw["grouping_state"] != poss_raw["grouping_state"].shift()).cumsum()

# drop the stoppage rows themselves -- dead time shouldn't count toward any event's duration,
# but the split they forced above is preserved
poss_raw_live = poss_raw[~poss_raw["is_stoppage"]].reset_index(drop=True)

possession_events = (
    poss_raw_live.groupby("event_id", as_index=False)
    .agg(
        possession_team=("possession_team_filled", "first"),
        start_frame=("frame_idx", "first"),
        end_frame=("frame_idx", "last"),
        n_frames=("frame_idx", "count"),
    )
)
possession_events["duration_sec"] = possession_events["n_frames"] / FPS
possession_events["pressing_team"] = 1 - possession_events["possession_team"]

print(f"{len(possession_events)} possession events (keep-last gap fill, split at stoppages >= {STOPPAGE_GAP_SEC}s)")
print(possession_events["duration_sec"].describe())

frame_to_event = poss_raw_live[["frame_idx", "event_id", "possession_team_filled"]]

event_frames = team_frames.merge(frame_to_event, on="frame_idx", how="inner")
# keep only rows where the pressing team actually matches this event's out-of-possession team --
# drops the rare frames where raw carrier_team briefly disagreed with the smoothed possession_team
event_frames = event_frames[event_frames["team"] != event_frames["possession_team_filled"]].copy()

event_pressure = (
    event_frames.groupby("event_id", as_index=False)
    .agg(
        mean_n_pressing=("n_pressing", "mean"),
        mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        pct_frames_under_pressure=("is_pressing", "mean"),
        n_frames_matched=("frame_idx", "count"),
    )
)

possession_events = possession_events.merge(event_pressure, on="event_id", how="left")
matched = possession_events["pct_frames_under_pressure"].notna().sum()
print(f"pressure computed for {matched} / {len(possession_events)} events "
      f"(unmatched events are ones where raw carrier_team never agreed with the filled possession_team)")

scored = possession_events.dropna(subset=["pct_frames_under_pressure"])

### 4.3 Turnovers and counter-press timing
A turnover is a change of carrier team between two **stable** holds (at least `TURNOVER_MIN_HOLD_SEC`), which filters 50/50 scrambles. Each turnover records:

- `loss_frame` — the losing team's last frame of its hold; `gain_frame` — the opponent's confirmed takeover frame (the real turnover boundary),
- `time_to_press_sec` — seconds from `loss_frame` to the losing team's first `is_pressing` frame within `MAX_SEARCH_WINDOW_SEC` (NaN if it never pressed),
- `press_response` — `fast` (within `COUNTER_PRESS_WINDOW_SEC`), `slow` (later, within the search window) or `never`,
- `regained_within_window` — did the losing team have the ball again within `COUNTER_PRESS_WINDOW_SEC` **after** `gain_frame`.

In [ ]:
# ---- stable carrier-team holds ----
raw_carrier = ball_frame_table.dropna(subset=["carrier_team"])[["frame_idx", "carrier_team"]].sort_values("frame_idx").reset_index(drop=True)
raw_carrier["state_change"] = (raw_carrier["carrier_team"] != raw_carrier["carrier_team"].shift()).cumsum()
carrier_runs = (
    raw_carrier.groupby("state_change")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"), carrier_team=("carrier_team", "first"))
    .reset_index(drop=True)
)
stable_runs = carrier_runs[carrier_runs["n_frames"] >= TURNOVER_MIN_HOLD_FRAMES].reset_index(drop=True)

# ---- one row per genuine turnover ----
turnover_rows = []
for i in range(len(stable_runs) - 1):
    losing_team = stable_runs.loc[i, "carrier_team"]
    gaining_team = stable_runs.loc[i + 1, "carrier_team"]
    if losing_team != gaining_team:
        turnover_rows.append({
            "losing_team": losing_team,
            "loss_frame": stable_runs.loc[i, "end_frame"],
            "gain_frame": stable_runs.loc[i + 1, "start_frame"],
        })
turnovers = pd.DataFrame(turnover_rows, columns=["losing_team", "loss_frame", "gain_frame"])

def seconds_to_first_press(team_id, loss_frame):
    window = press_lookup[
        (press_lookup["team"] == team_id) &
        (press_lookup["frame_idx"] >= loss_frame) &
        (press_lookup["frame_idx"] <= loss_frame + MAX_SEARCH_WINDOW_FRAMES) &
        (press_lookup["is_pressing"])
    ]
    return (window["frame_idx"].min() - loss_frame) / FPS if len(window) > 0 else np.nan

def classify_press_response(t):
    if pd.isna(t):
        return "never"
    elif t <= COUNTER_PRESS_WINDOW_SEC:
        return "fast"
    else:
        return "slow"

turnovers["time_to_press_sec"] = [seconds_to_first_press(t, f) for t, f in zip(turnovers["losing_team"], turnovers["loss_frame"])]
turnovers["press_response"] = pd.Categorical(
    turnovers["time_to_press_sec"].apply(classify_press_response), categories=["fast", "slow", "never"], ordered=True
)
# regain is searched strictly AFTER the opponent's takeover frame (searching from the losing team's own last touch matched trivially)
turnovers["regained_within_window"] = [
    team_had_ball_between(t, g + 1, g + COUNTER_PRESS_WINDOW_FRAMES)
    for t, g in zip(turnovers["losing_team"], turnovers["gain_frame"])
]

print(f"{len(turnovers)} genuine turnovers | "
      f"press detected within {MAX_SEARCH_WINDOW_SEC}s on {int(turnovers['time_to_press_sec'].notna().sum())}")

### 4.4 Passes and defensive actions
Passes are detected directly from `carrier_track_id` transitions: a same-team handover is a completed pass and a change of team is an interception. Transitions shorter than `MIN_PASS_DISTANCE_M` are duels (a different-team one is logged as a **tackle**), gaps longer than `MAX_RECEPTION_GAP_FRAMES` are skipped here (they are picked up in 4.5), and goalkeeper claims are excluded.

`was_under_pressure(team, frame)` is true when the **opponent** had `is_pressing` for `MIN_SUSTAINED_PRESSURE_FRAMES` **consecutive** frames within `PRESSURE_LOOKBACK_SEC` before the release. A single flickering frame does not count.

In [ ]:
def was_under_pressure(passer_team, release_frame):
    passer_team = int(passer_team)
    opponent = 1 - passer_team
    release_frame = int(release_frame)
    consecutive = 0
    for f in range(max(0, release_frame - PRESSURE_LOOKBACK_FRAMES), release_frame + 1):
        if press_flags.get((f, opponent), False):
            consecutive += 1
            if consecutive >= MIN_SUSTAINED_PRESSURE_FRAMES:
                return True
        else:
            consecutive = 0
    return False

# --- detect passes + defensive actions from scratch, from carrier_track_id transitions ---

carrier_seq = (
    ball_frame_table.dropna(subset=["carrier_track_id"])
    [["frame_idx", "carrier_track_id", "carrier_team"]]
    .sort_values("frame_idx")
    .reset_index(drop=True)
)
carrier_seq["run_id"] = (carrier_seq["carrier_track_id"] != carrier_seq["carrier_track_id"].shift()).cumsum()

carrier_runs_full = (
    carrier_seq.groupby("run_id")
    .agg(
        track_id=("carrier_track_id", "first"),
        team=("carrier_team", "first"),
        start_frame=("frame_idx", "first"),
        end_frame=("frame_idx", "last"),
        n_frames=("frame_idx", "count"),
    )
    .reset_index(drop=True)
)
carrier_runs_full = carrier_runs_full[carrier_runs_full["n_frames"] >= MIN_HOLD_FRAMES_PASS].reset_index(drop=True)

pos_lookup = player_frame_table.set_index(["frame_idx", "track_id"])[["pitch_x", "pitch_y"]]
role_lookup = player_frame_table.set_index(["frame_idx", "track_id"])["role"]

candidates = []
defensive_actions = []

for i in range(len(carrier_runs_full) - 1):
    passer = carrier_runs_full.loc[i]
    receiver = carrier_runs_full.loc[i + 1]

    gap_frames = receiver["start_frame"] - passer["end_frame"]
    if gap_frames > MAX_RECEPTION_GAP_FRAMES:
        continue  # too long a gap -- ball likely went out of play, not a resolved event

    try:
        passer_pos = pos_lookup.loc[(passer["end_frame"], passer["track_id"])]
        receiver_pos = pos_lookup.loc[(receiver["start_frame"], receiver["track_id"])]
    except KeyError:
        continue

    same_team = receiver["team"] == passer["team"]
    receiver_role = role_lookup.get((receiver["start_frame"], receiver["track_id"]), None)

    if (not same_team) and receiver_role == "goalkeeper":
        continue  # save/claim by the keeper -- not a midfield defensive action, exclude entirely

    dist_m = float(np.hypot(receiver_pos["pitch_x"] - passer_pos["pitch_x"],
                             receiver_pos["pitch_y"] - passer_pos["pitch_y"]))

    if dist_m < MIN_PASS_DISTANCE_M:
        if not same_team:
            defensive_actions.append({
                "defending_team": int(receiver["team"]),
                "attacking_team": int(passer["team"]),
                "action_type": "tackle",
                "frame": int(passer["end_frame"]),
                "x": float(passer_pos["pitch_x"]),
                "y": float(passer_pos["pitch_y"]),
            })
        continue  # same-team short touch = noise; different-team short = captured above as a tackle

    if not same_team:
        defensive_actions.append({
            "defending_team": int(receiver["team"]),
            "attacking_team": int(passer["team"]),
            "action_type": "interception",
            "frame": int(passer["end_frame"]),
            "x": float(passer_pos["pitch_x"]),
            "y": float(passer_pos["pitch_y"]),
        })

    candidates.append({
        "passer_track_id": passer["track_id"],
        "passer_team": int(passer["team"]),
        "receiver_track_id": receiver["track_id"],
        "receiver_team": int(receiver["team"]),
        "release_frame": int(passer["end_frame"]),
        "reception_frame": int(receiver["start_frame"]),
        "release_x": float(passer_pos["pitch_x"]),
        "release_y": float(passer_pos["pitch_y"]),
        "pass_distance_m": dist_m,
        "outcome": "completed" if same_team else "intercepted",
    })

pass_table = pd.DataFrame(candidates)
defensive_actions = pd.DataFrame(defensive_actions)

print(f"{len(pass_table)} pass attempts detected (from {len(carrier_runs_full)} carrier runs)")
print(pass_table["outcome"].value_counts())
print(f"\n{len(defensive_actions)} defensive actions captured (goalkeeper saves/claims excluded)")
print(defensive_actions["action_type"].value_counts())

pass_table["under_pressure"] = [was_under_pressure(t, f) for t, f in zip(pass_table["passer_team"], pass_table["release_frame"])]
pass_table["success"] = pass_table["outcome"] == "completed"

### 4.5 Pass outcomes
Every resolved carrier transition is classified into what actually happened to the build-up: `progressed_forward`, `forced_sideways`, `forced_backward`, `lost_intercepted`, `lost_tackled`, `lost_after_long_ball`, `retained_after_gap` or `unresolved`. Passes, tackles and long-gap transitions are combined into one table, `outcome_events`, with the zone of the action (from the acting team's perspective).

In [ ]:
def receiver_position(row):
    try:
        pos = pos_lookup.loc[(row["reception_frame"], row["receiver_track_id"])]
        return pos["pitch_x"], pos["pitch_y"]
    except KeyError:
        return np.nan, np.nan

recv_xy = pass_table.apply(receiver_position, axis=1, result_type="expand")
pass_table["receiver_x"] = recv_xy[0]
pass_table["receiver_y"] = recv_xy[1]

def progression_m(row):
    # positive = net advance toward the attacking goal, negative = net retreat, direction-corrected per team
    if pd.isna(row["receiver_x"]):
        return np.nan
    if row["passer_team"] == 0:
        return row["receiver_x"] - row["release_x"]
    else:
        return row["release_x"] - row["receiver_x"]

pass_table["progression_m"] = pass_table.apply(progression_m, axis=1)

def classify_pass_outcome(row):
    if row["outcome"] == "intercepted":
        return "lost_intercepted"
    if pd.isna(row["progression_m"]):
        return "unresolved"  # receiver position missing from player_frame_table at reception frame
    if row["progression_m"] >= FORWARD_PROGRESS_THRESH_M:
        return "progressed_forward"
    elif row["progression_m"] <= BACKWARD_THRESH_M:
        return "forced_backward"
    else:
        return "forced_sideways"

pass_table["pass_outcome_detail"] = pass_table.apply(classify_pass_outcome, axis=1)

print("pass outcome breakdown, under pressure vs not, by team:")
show(pass_table.groupby(["passer_team", "under_pressure", "pass_outcome_detail"]).size().unstack(fill_value=0), digits=0)

# ---- tackles: short-distance different-team transitions (excluded from pass_table) ----
tackle_outcomes = defensive_actions[defensive_actions["action_type"] == "tackle"][
    ["attacking_team", "frame", "x"]
].rename(columns={"attacking_team": "team"}).copy()
tackle_outcomes["under_pressure"] = [was_under_pressure(t, f) for t, f in zip(tackle_outcomes["team"], tackle_outcomes["frame"])]
tackle_outcomes["pass_outcome_detail"] = "lost_tackled"
tackle_outcomes["zone"] = [zone_for_team(x, t) for x, t in zip(tackle_outcomes["x"], tackle_outcomes["team"])]
tackle_outcomes = tackle_outcomes.drop(columns=["frame", "x"])

# ---- long-gap transitions (longer than MAX_RECEPTION_GAP_FRAMES): throw-ins, stoppages, loose-ball scrambles ----
long_gap_records = []
for i in range(len(carrier_runs_full) - 1):
    passer = carrier_runs_full.loc[i]
    receiver = carrier_runs_full.loc[i + 1]
    gap_frames = receiver["start_frame"] - passer["end_frame"]
    if gap_frames <= MAX_RECEPTION_GAP_FRAMES:
        continue  # already handled as a resolved pass / interception / tackle

    try:
        release_x = pos_lookup.loc[(passer["end_frame"], passer["track_id"])]["pitch_x"]
    except KeyError:
        release_x = np.nan

    same_team = receiver["team"] == passer["team"]
    long_gap_records.append({
        "team": int(passer["team"]),
        "under_pressure": was_under_pressure(passer["team"], passer["end_frame"]),
        "pass_outcome_detail": "retained_after_gap" if same_team else "lost_after_long_ball",
        "zone": zone_for_team(release_x, int(passer["team"])) if pd.notna(release_x) else None,
    })
long_gap_events = pd.DataFrame(long_gap_records, columns=["team", "under_pressure", "pass_outcome_detail", "zone"])

# ---- unified table: one row per resolved carrier transition ----
pass_outcomes = pass_table[["passer_team", "under_pressure", "pass_outcome_detail", "release_x"]].rename(columns={"passer_team": "team"}).copy()
pass_outcomes["zone"] = [zone_for_team(x, t) for x, t in zip(pass_outcomes["release_x"], pass_outcomes["team"])]
pass_outcomes = pass_outcomes.drop(columns="release_x")

outcome_events = pd.concat([pass_outcomes, tackle_outcomes, long_gap_events], ignore_index=True)
outcome_events["pass_outcome_detail"] = pd.Categorical(
    outcome_events["pass_outcome_detail"], categories=FINAL_OUTCOME_ORDER, ordered=True
)

print(f"{len(pass_outcomes)} passes + {len(tackle_outcomes)} tackles + {len(long_gap_events)} long-gap transitions "
      f"= {len(outcome_events)} outcome events")

### 4.6 Reception triggers
For every completed pass, describe the receiver's situation at the moment of reception (proxies for the classic press triggers, since there is no body-orientation model): near the touchline, deep in own third, received a backward pass, isolated (few open lanes), slow first touch. Then check whether the opponent **pressed within `PRESS_ONSET_WINDOW_SEC`** of the reception (`press_followed`).

In [ ]:
reception_triggers = pass_table[pass_table["outcome"] == "completed"][
    ["receiver_track_id", "passer_team", "reception_frame", "receiver_x", "receiver_y", "progression_m"]
].rename(columns={"passer_team": "team"}).dropna(subset=["receiver_x", "receiver_y"]).copy()

reception_triggers["zone"] = reception_triggers.apply(lambda r: zone_for_team(r["receiver_x"], r["team"]), axis=1)
reception_triggers["dist_to_touchline_m"] = np.minimum(
    reception_triggers["receiver_y"], PITCH_WIDTH - reception_triggers["receiver_y"]
)

reception_triggers["trigger_touchline"] = reception_triggers["dist_to_touchline_m"] < TOUCHLINE_DIST_THRESH_M
reception_triggers["trigger_deep_own_third"] = reception_triggers["zone"] == "own_third"
reception_triggers["trigger_backward_pass"] = reception_triggers["progression_m"] < 0

# open passing lanes at the reception frame, reusing frame_lane_summary from 3.2
lane_lookup = frame_lane_summary.set_index("frame_idx")["n_lanes_open"]
reception_triggers["n_lanes_open"] = reception_triggers["reception_frame"].map(lane_lookup)
reception_triggers["trigger_isolated"] = reception_triggers["n_lanes_open"] < ISOLATED_LANES_THRESH

print(f"{reception_triggers['n_lanes_open'].isna().sum()} receptions with no lane data at that frame (dropped from trigger_isolated only)")

# --- first-touch speed: how quickly the receiver moved the ball in the moment right after control ---

def first_touch_speed(row):
    try:
        p0 = pos_lookup.loc[(row["reception_frame"], row["receiver_track_id"])]
        p1 = pos_lookup.loc[(row["reception_frame"] + FIRST_TOUCH_WINDOW_FRAMES, row["receiver_track_id"])]
    except KeyError:
        return np.nan
    dist_m = float(np.hypot(p1["pitch_x"] - p0["pitch_x"], p1["pitch_y"] - p0["pitch_y"]))
    return dist_m / (FIRST_TOUCH_WINDOW_FRAMES / FPS)

reception_triggers["first_touch_speed_mps"] = reception_triggers.apply(first_touch_speed, axis=1)
reception_triggers["trigger_static_touch"] = reception_triggers["first_touch_speed_mps"] < STATIC_TOUCH_SPEED_MPS

print(f"{reception_triggers['first_touch_speed_mps'].isna().sum()} receptions with no first-touch speed "
      f"(receiver missing from player_frame_table {FIRST_TOUCH_WINDOW_FRAMES} frames later -- likely a pass near a stoppage)")

# --- did opponent pressure actually follow this reception, within PRESS_ONSET_WINDOW_SEC? ---
def press_follows_reception(row):
    team = int(row["team"])
    opponent = 1 - team
    frame0 = int(row["reception_frame"])
    for f in range(frame0, frame0 + PRESS_ONSET_WINDOW_FRAMES + 1):
        if press_flags.get((f, opponent), False):
            return True
    return False

reception_triggers["press_followed"] = reception_triggers.apply(press_follows_reception, axis=1)

## 5. Metrics

Each cell below computes one metric family and prints one compact table. **Credit** says which team the number describes. The full definitions live in the catalogue (6.1).

### 5.1 Pressing time share  ·  `press_time_pct`  ·  credit: pressing team
Seconds with `is_pressing` ÷ live out-of-possession seconds (stoppages excluded).

In [ ]:
total_pressure_frames_by_team = team_pressure_events.groupby("team")["n_frames"].sum()
total_pressure_seconds_by_team = total_pressure_frames_by_team / FPS

bft_live = ball_frame_table.merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
bft_live["is_stoppage"] = bft_live["is_stoppage"].fillna(False)
bft_live = bft_live[~bft_live["is_stoppage"]]  # dead time is not live play

# Denominator: seconds a team is OUT of possession on frames where a ball carrier is identified (the OTHER team has the ball).
# Pressing can only be detected on frames with an identified carrier, so numerator and denominator must use the same frames.
out_of_possession_seconds = {t: (bft_live["carrier_team"] == 1 - t).sum() / FPS for t in (0, 1)}
# reference only: out-of-possession seconds on every frame that has a possession label (includes frames without a carrier)
out_of_possession_seconds_labelled = {
    t: (bft_live["possession_team"].notna().sum() - (bft_live["possession_team"] == t).sum()) / FPS for t in (0, 1)
}

match_duration_sec = (player_frame_table["frame_idx"].max() + 1) / FPS
live_sec = len(bft_live) / FPS

pressing_time_pct = {
    t: _num(total_pressure_seconds_by_team.get(t, 0) / out_of_possession_seconds[t] * 100) if out_of_possession_seconds[t] else None
    for t in (0, 1)
}

press_time_table = pd.DataFrame(
    {
        "pressing_sec": [total_pressure_seconds_by_team.get(t, 0) for t in (0, 1)],
        "out_of_possession_sec": [out_of_possession_seconds[t] for t in (0, 1)],
        "press_time_pct": [pressing_time_pct[t] for t in (0, 1)],
        "oop_sec_all_labelled_frames": [out_of_possession_seconds_labelled[t] for t in (0, 1)],
    },
    index=[TEAM_NAMES[0], TEAM_NAMES[1]],
)
print(f"match {match_duration_sec:.1f}s | stoppages excluded {stoppages['duration_sec'].sum():.1f}s | live {live_sec:.1f}s")
print("press_time_pct = pressing_sec / out_of_possession_sec, where out-of-possession time counts only frames with an identified carrier\n")
print(press_time_table.round(1).to_string())

### 5.2 Pressure by zone and zone-weighted intensity  ·  `zone_weighted_press_pct`  ·  credit: pressing team
Pressing frames split by the third of the pitch the ball is in (from the pressing team's perspective). The weighted score counts own third ×1, middle ×2, attacking ×3 and is normalised by the theoretical maximum (every out-of-possession frame pressing in the attacking third), so it is bounded 0–100.

In [ ]:
team_frames_zone = team_pressure_full.copy()
team_frames_zone["press_zone"] = [zone_for_team(x, t) for x, t in zip(team_frames_zone["press_x"], team_frames_zone["team"])]

pressing_only = team_frames_zone[team_frames_zone["is_pressing"] & team_frames_zone["press_zone"].notna()]

zone_counts = pressing_only.groupby(["team", "press_zone"]).size().unstack(fill_value=0)
zone_pct = zone_counts.div(zone_counts.sum(axis=1), axis=0) * 100

show(zone_counts, "pressure frames by team and zone:", 0)
show(zone_pct, "\n% distribution within each team's own pressing:")

# ---- zone-weighted intensity ----
weighted = team_frames_zone[team_frames_zone["is_pressing"] & team_frames_zone["press_zone"].notna()].copy()
weighted["zone_weight"] = weighted["press_zone"].map(ZONE_WEIGHTS)

weighted_pressure_frames = weighted.groupby("team")["zone_weight"].sum()
weighted_pressure_seconds = weighted_pressure_frames / FPS

zone_weighted_pct_of_max = {
    team_id: _num(
        weighted_pressure_seconds.get(team_id, 0)
        / (out_of_possession_seconds[team_id] * ZONE_WEIGHTS["attacking_third"])
        * 100
    )
    for team_id in [0, 1]
}

show(weighted_pressure_seconds, "\nweighted pressure seconds by team (own=1x, middle=2x, attacking=3x):")
print("\nzone-weighted pressing intensity, % of theoretical max:")
for team_id in [0, 1]:
    v = zone_weighted_pct_of_max[team_id]
    print(f"{TEAM_NAMES[team_id]}: " + ("n/a" if v is None else f"{v:.1f}%"))

### 5.3 Defensive line height  ·  `defensive_line_m`  ·  credit: defending team
Distance from own goal of the last-1 and last-3 defenders, per frame, with IQR outlier removal per team. Uses the clean `outfield` table from 2.3.

In [ ]:
def compute_line_height(df, n_defenders):
    return (
        df.sort_values("dist_from_own_goal")
        .groupby(["frame_idx", "team"])
        .head(n_defenders)
        .groupby(["frame_idx", "team"], as_index=False)["dist_from_own_goal"]
        .mean()
        .rename(columns={"dist_from_own_goal": "line_height_m"})
    )

def remove_outliers_iqr(df, col, group_cols):
    parts = []
    for _, g in df.groupby(group_cols):
        q1, q3 = g[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        parts.append(g[(g[col] >= lo) & (g[col] <= hi)])
    return pd.concat(parts, ignore_index=True)

line_last1 = compute_line_height(outfield, 1).merge(poss_by_frame, on="frame_idx", how="left")
line_last3 = compute_line_height(outfield, 3).merge(poss_by_frame, on="frame_idx", how="left")

line_last1_clean = remove_outliers_iqr(line_last1, "line_height_m", ["team"])
line_last3_clean = remove_outliers_iqr(line_last3, "line_height_m", ["team"])

avg_line_height = line_last3_clean.groupby("team")["line_height_m"].mean()

show(line_last1_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]), "defensive line, last-1 defender (m from own goal):")
show(line_last3_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]), "\ndefensive line, last-3 defenders average (m from own goal):")

### 5.4 Share of opponent possession under pressure  ·  `press_share_pct`  ·  credit: pressing team
Per possession event (4.2), the share of frames the possessing team was pressed; then averaged per **pressing** team, **weighted by possession duration** (a 40 s press counts more than a 2 s one). `duration_weighted_pct_under_pressure` is the headline; the simple mean is kept only as a diagnostic.

In [ ]:
final_pressure_by_team = (
    scored.groupby("pressing_team")
    .apply(lambda g: pd.Series({
        "duration_weighted_pct_under_pressure": np.average(g["pct_frames_under_pressure"], weights=g["duration_sec"]) * 100,
        "simple_mean_pct_under_pressure": g["pct_frames_under_pressure"].mean() * 100,
        "n_events": len(g),
        "total_duration_sec": g["duration_sec"].sum(),
    }))
)
print("\nfinal pressure by pressing team (duration-weighted mean is the headline number):")
show(final_pressure_by_team)

### 5.5 PPDA  ·  `ppda`  ·  credit: defending team
Opponent completed passes outside the defending team's **own** defensive third ÷ the defending team's own defensive actions (interceptions and tackles) outside that same zone. Both parts use the defending team's zone convention. Lower = more aggressive pressing.

In [ ]:
ppda_by_team = {}
ppda_detail = {}

for team_id in [0, 1]:
    opponent = 1 - team_id

    # numerator: opponent's completed passes, classified in THIS team's own zone convention --
    # excluded only when they happen in this team's own defensive third (last-ditch territory)
    opp_completed = pass_table[
        (pass_table["outcome"] == "completed") &
        (pass_table["passer_team"] == opponent)
    ].copy()
    opp_completed["zone"] = opp_completed["release_x"].apply(lambda x: zone_for_team(x, team_id))
    passes_allowed = int((opp_completed["zone"] != "own_third").sum())

    # denominator: this team's own defensive actions, same zone convention
    own_actions = defensive_actions[defensive_actions["defending_team"] == team_id].copy()
    own_actions["zone"] = own_actions["x"].apply(lambda x: zone_for_team(x, team_id))
    actions_made = int((own_actions["zone"] != "own_third").sum())

    ppda_by_team[team_id] = passes_allowed / actions_made if actions_made > 0 else np.nan
    ppda_detail[team_id] = {"passes_allowed": passes_allowed, "actions_made": actions_made}

print("passes allowed / defensive actions made, by team (excluding each team's own defensive third):")
for team_id in [0, 1]:
    print(f"{TEAM_NAMES[team_id]}: {ppda_detail[team_id]}")

print("\nPPDA by team (lower = more aggressive pressing):")
for team_id, val in ppda_by_team.items():
    print(f"{TEAM_NAMES[int(team_id)]}: {val:.2f}")

### 5.6 Co-press and press → regain  ·  `coordinated_press_pct`, `press_regain_rate_pct`  ·  credit: pressing team
- **Coordinated press:** share of pressure events where at least two defenders pressed at once (`max_n_pressing >= 2`).
- **Press → regain:** share of pressure events after which the pressing team had the ball within `MAX_REGAIN_SEC` of the event ending.

In [ ]:
print("pressure event duration (s), by team:")
show(team_pressure_events.groupby("team")["duration_sec"].describe())

print("\nmax simultaneous pressers per event, by team:")
show(team_pressure_events.groupby("team")["max_n_pressing"].describe())

coordinated_press_share_pct = (
    team_pressure_events.groupby("team")["max_n_pressing"].apply(lambda s: (s >= 2).mean() * 100).round(1)
)
print("\nshare of events that were a coordinated press (max_n_pressing >= 2), by team (%):")
show(coordinated_press_share_pct)

regain_summary = (
    team_pressure_events.groupby("team")["regained_ball"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "regain_rate"})
)
regain_summary["regain_rate"] = (regain_summary["regain_rate"] * 100).round(1)
print(f"\npress -> regain rate (ball won back by the pressing team within {MAX_REGAIN_SEC}s of the event ending):")
show(regain_summary)

### 5.7 Time-to-press and counter-press  ·  `time_to_press_median_sec`, `counter_press_fast_regain_pct`  ·  credit: team that lost the ball
How quickly does a team press after **losing** the ball, and does pressing fast actually predict winning it back? Turnovers come from 4.3; `losing_team` is the team whose counter-press is being measured.

In [ ]:
time_to_press_stats = turnovers.groupby("losing_team")["time_to_press_sec"].agg(["mean", "median", "min", "max", "count"])
print("time-to-press after losing the ball, by team (seconds; only turnovers where a press was detected):")
show(time_to_press_stats)

print(f"\ncounter-press regain rate by team (ball regained within {COUNTER_PRESS_WINDOW_SEC}s of the opponent's takeover, regardless of press):")
counter_press_regain_by_team = turnovers.groupby("losing_team")["regained_within_window"].agg(["mean", "sum", "count"])
counter_press_regain_by_team["mean"] = (counter_press_regain_by_team["mean"] * 100).round(1)
show(counter_press_regain_by_team.rename(columns={"mean": "regain_rate_pct"}))

counter_press_summary = (
    turnovers.groupby(["losing_team", "press_response"], observed=False)["regained_within_window"]
    .agg(regains="sum", turnovers="count")
    .reset_index()
)
counter_press_summary["regain_rate_pct"] = (counter_press_summary["regains"] / counter_press_summary["turnovers"] * 100).round(1)

print("=" * 70)
print("COUNTER-PRESS EFFECTIVENESS BY PRESS-RESPONSE SPEED")
print("=" * 70)
print(f"fast  = pressed within {COUNTER_PRESS_WINDOW_SEC:.0f}s of the opponent's confirmed takeover")
print(f"slow  = pressed between {COUNTER_PRESS_WINDOW_SEC:.0f}-{MAX_SEARCH_WINDOW_SEC}s (the search window from 4.3)")
print(f"never = no press detected within the {MAX_SEARCH_WINDOW_SEC}s search window from 4.3 at all")
print()

for team_id, team_name in TEAM_NAMES.items():
    print(f"{team_name}")
    print("-" * 40)
    team_rows = (
        counter_press_summary[counter_press_summary["losing_team"] == team_id]
        .set_index("press_response")
        .reindex(["fast", "slow", "never"])
    )
    for response, row in team_rows.iterrows():
        n = 0 if pd.isna(row["turnovers"]) else int(row["turnovers"])
        if n == 0:
            print(f"  {response:<6} : n/a (0 turnovers)")
            continue
        regains = int(row["regains"])
        rate = row["regain_rate_pct"]
        print(f"  {response:<6} : {rate:5.1f}% regain rate  ({regains}/{n} turnovers)")
    print()

print("=" * 70)
print("OVERALL (both teams combined)")
print("=" * 70)
overall = (
    turnovers.groupby("press_response", observed=False)["regained_within_window"]
    .agg(regains="sum", turnovers="count")
    .reindex(["fast", "slow", "never"])
)
overall["regain_rate_pct"] = (overall["regains"] / overall["turnovers"] * 100).round(1)
for response, row in overall.iterrows():
    n = int(row["turnovers"])
    print(f"  {response:<6} : {row['regain_rate_pct']:5.1f}% regain rate  ({int(row['regains'])}/{n} turnovers)")

### 5.8 Own passing under pressure  ·  `passes_under_pressure_pct`, `pass_success_under_pressure_pct`  ·  credit: team in possession
How often a team's own passes were made under pressure (4.4), and how often those passes were completed vs passes made without pressure.

In [ ]:
print("passes by team and pressure state:")
show(pass_table.groupby(["passer_team", "under_pressure"]).size().rename("n_passes"), digits=0)

pass_summary = (
    pass_table.groupby(["passer_team", "under_pressure"])["success"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "success_rate"})
)
pass_summary["success_rate"] = (pass_summary["success_rate"] * 100).round(1)
show(pass_summary, "\npass success rate by team, under pressure vs not:")

print("\ntotal passes attempted while under pressure, by team:")
show(pass_table[pass_table["under_pressure"]].groupby("passer_team").size().rename("n_passes_under_pressure"), digits=0)

### 5.9 Pass outcomes and disruption  ·  `disruption_forced_pct`, `pressed_turnover_rate_<zone>_pct`  ·  credit: pressing team
What the press does to the **opponent's** build-up. `outcome_events` (4.5) is indexed by the team whose action it is, so `disruption_rate` and `zone_summary` describe the **pressed** team and the credit belongs to the opponent's press. The team profile (6.2) re-orients them to the pressing team.

- **Disruption rate:** share of a team's pressured actions that did not progress cleanly (forced sideways or backward, intercepted, tackled, or lost on a long ball).
- **Turnover rate:** share of a team's pressured actions that ended in a lost ball.

In [ ]:
outcome_counts = (
    outcome_events.groupby(["team", "under_pressure", "pass_outcome_detail"], observed=False)
    .size()
    .unstack(fill_value=0)
)
print("carrier-transition outcomes by team and pressure state (counts):")
show(outcome_counts, digits=0)

outcome_pct = outcome_counts.div(outcome_counts.sum(axis=1), axis=0) * 100
print("\nsame table as % within each team/pressure-state row:")
show(outcome_pct)

# headline: of the carrier actions a team made WHILE UNDER PRESSURE, what share were disrupted
# in some way (not clean forward progression) -- this is the opponent's press-effectiveness number,
# so index it by the team that got disrupted and remember the credit belongs to (1 - team)
pressured = outcome_events[outcome_events["under_pressure"] == True].copy()
pressured["disrupted"] = pressured["pass_outcome_detail"].isin(DISRUPTIVE)
disruption_rate = pressured.groupby("team")["disrupted"].mean() * 100

print("\n% of a team's own pressured carrier actions that were disrupted in some way")
print("(credit for the disruption belongs to the OPPOSING team's press):")
show(disruption_rate.rename("pct_disrupted"))

# ---- by zone (zone is relative to the team whose action it is) ----
outcome_events_zoned = outcome_events.dropna(subset=["zone"])
print(f"\n{outcome_events['zone'].isna().sum()} events with unresolvable zone (missing position), dropped from the zone breakdown")

# --- disruption rate and turnover rate, by team and zone, pressured actions only ---

pressured_z = outcome_events_zoned[outcome_events_zoned["under_pressure"] == True].copy()
pressured_z["disrupted"] = pressured_z["pass_outcome_detail"].isin(DISRUPTIVE)
pressured_z["turned_over"] = pressured_z["pass_outcome_detail"].isin(
    ["lost_intercepted", "lost_tackled", "lost_after_long_ball"]
)

zone_summary = (
    pressured_z.groupby(["team", "zone"])
    .agg(n_events=("disrupted", "size"), disruption_rate=("disrupted", "mean"), turnover_rate=("turned_over", "mean"))
)
zone_summary["disruption_rate"] = (zone_summary["disruption_rate"] * 100).round(1)
zone_summary["turnover_rate"] = (zone_summary["turnover_rate"] * 100).round(1)
zone_summary = zone_summary.reindex(pd.MultiIndex.from_product([[0, 1], ZONE_ORDER], names=["team", "zone"]))

print("of a team's own pressured actions in each zone -- disruption rate and turnover rate")
print("(zone is relative to the team whose action this is; credit for the disruption/turnover belongs to the opponent's press):")
show(zone_summary)

### 5.10 Pressing triggers  ·  `press_follow_rate_pct`  ·  credit: pressing team
How often a press actually follows a reception, overall and when a trigger condition holds (touchline, deep own third, backward pass, isolated, static first touch).

In [ ]:
TRIGGER_COLS = ["trigger_touchline", "trigger_deep_own_third", "trigger_backward_pass",
                "trigger_isolated", "trigger_static_touch"]

baseline_rate = reception_triggers["press_followed"].mean() * 100
print(f"baseline: a press follows any reception {baseline_rate:.1f}% of the time (n={len(reception_triggers)})\n")

for col in TRIGGER_COLS:
    with_trigger = reception_triggers[reception_triggers[col] == True]
    without_trigger = reception_triggers[reception_triggers[col] == False]
    print(f"{col}: press followed {with_trigger['press_followed'].mean()*100:.1f}% of the time "
          f"(n={len(with_trigger)}) vs {without_trigger['press_followed'].mean()*100:.1f}% otherwise")

print("\ntrigger prevalence and press-follow rate by receiving team (%):")
show(reception_triggers.groupby("team")[TRIGGER_COLS + ["press_followed"]].mean() * 100)

### 5.11 Team shape  ·  `compactness_pressing_m2`, `compactness_baseline_m2`  ·  credit: defending team
Out-of-possession depth (back-3 to front-3 along the team's attacking direction), width (IQR-trimmed lateral spread) and compactness area = depth × width. Compares the shape **while the team is pressing** vs its resting out-of-possession shape.

In [ ]:
_shape_base = outfield.merge(poss_by_frame, on="frame_idx", how="left")
_shape_base = _shape_base[
    _shape_base["possession_team"].notna() & (_shape_base["team"] != _shape_base["possession_team"])
]

def _team_frame_depth(df):
    d = df.sort_values("dist_from_own_goal")
    n = len(d)
    if n < MIN_PLAYERS_FOR_SHAPE:
        return pd.Series({"back_line_m": np.nan, "front_line_m": np.nan, "depth_m": np.nan, "n_players": n})
    back = d.head(BACK_N)["dist_from_own_goal"].mean()
    front = d.tail(FRONT_N)["dist_from_own_goal"].mean()
    return pd.Series({"back_line_m": back, "front_line_m": front, "depth_m": front - back, "n_players": n})

def _team_frame_width(y_vals):
    y = np.asarray(y_vals)
    if len(y) < 4:
        return np.nan
    q1, q3 = np.percentile(y, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - WIDTH_IQR_MULT * iqr, q3 + WIDTH_IQR_MULT * iqr
    y_trimmed = y[(y >= lo) & (y <= hi)]
    return float(y_trimmed.max() - y_trimmed.min()) if len(y_trimmed) else np.nan

depth_per_frame = _shape_base.groupby(["frame_idx", "team"]).apply(_team_frame_depth).reset_index()
depth_per_frame = depth_per_frame.dropna(subset=["depth_m"])

width_per_frame = (
    _shape_base.groupby(["frame_idx", "team"])["pitch_y"]
    .apply(_team_frame_width)
    .reset_index(name="width_m")
)

team_shape = depth_per_frame.merge(width_per_frame, on=["frame_idx", "team"], how="inner")
team_shape["compactness_area_m2"] = team_shape["depth_m"] * team_shape["width_m"]

print("out-of-possession team shape, by team (smaller area = more compact block):")
show(team_shape.groupby("team")[["depth_m", "width_m", "compactness_area_m2"]].agg(["mean", "median", "std"]))

# --- does a team actually get tighter when it's actively pressing, or is shape constant regardless? ---
team_shape_pressure = team_shape.merge(
    team_frames[["frame_idx", "team", "is_pressing"]], on=["frame_idx", "team"], how="left"
)

print("compactness area (m^2) while actively pressing vs not, by team:")
show(
    team_shape_pressure.groupby(["team", "is_pressing"])["compactness_area_m2"]
    .agg(["mean", "median", "count"])
)

### 5.12 Pressing timeline  ·  credit: pressing team
Share of live out-of-possession frames with `is_pressing`, in `BIN_MINUTES` bins, stoppages excluded. Bins with fewer than `MIN_BIN_FRAMES` frames are masked (NaN) because one short sequence would swing them. A short trailing bin (less than `MIN_LAST_BIN_FRAC` of a full bin) is merged into the previous one and labelled with the true end time, e.g. `40-46:53min`.

In [ ]:
live_team_frames = team_frames.merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
live_team_frames["is_stoppage"] = live_team_frames["is_stoppage"].fillna(False)
live_team_frames = live_team_frames[~live_team_frames["is_stoppage"]].copy()
live_team_frames["time_bin"] = assign_time_bins(live_team_frames["frame_idx"])
last_time_bin = live_team_frames["time_bin"].max()

pressing_by_bin = (
    live_team_frames.groupby(["team", "time_bin"])["is_pressing"]
    .mean()
    .reset_index()
    .rename(columns={"is_pressing": "pct_pressing"})
)
pressing_by_bin["pct_pressing"] = (pressing_by_bin["pct_pressing"] * 100).round(1)
pressing_by_bin["bin_label"] = pressing_by_bin["time_bin"].apply(lambda b: bin_label(b, last_time_bin))

pressing_intensity_timeline = (
    pressing_by_bin
    .sort_values("time_bin")
    .pivot(index="bin_label", columns="team", values="pct_pressing")
)
ordered_labels = pressing_by_bin.sort_values("time_bin")["bin_label"].unique()
pressing_intensity_timeline = pressing_intensity_timeline.loc[ordered_labels]

# ---- mask low-sample bins ----
bin_frame_counts = live_team_frames.groupby(["team", "time_bin"]).size().reset_index(name="n_frames")
bin_frame_counts["bin_label"] = bin_frame_counts["time_bin"].apply(lambda b: bin_label(b, last_time_bin))
low_sample = bin_frame_counts[bin_frame_counts["n_frames"] < MIN_BIN_FRAMES]

for _, row in low_sample.iterrows():
    pressing_intensity_timeline.loc[row["bin_label"], row["team"]] = np.nan

print(f"masked {len(low_sample)} team/bin cell(s) below MIN_BIN_FRAMES={MIN_BIN_FRAMES} (unreliable sample size)")
show(pressing_intensity_timeline, "pressing timeline (% of live out-of-possession frames with an identified carrier that are pressing):")

### 5.13 Possession  ·  `possession_pct_carrier`  ·  credit: team in possession
How much of the live game each team has the ball, where on the pitch it has it, and how long each spell lasts. Stoppages (2.2) are excluded.

- **Possession share (two versions).** `possession_pct_labelled` uses every live frame with a possession label (gaps filled with the last known team). `possession_pct_carrier` uses only frames where a ball carrier is identified. Quote the carrier-based figure when comparing with the pressing metrics, since both then use the same frames.
- **Possession events.** Count and length (mean, median, longest) of possession spells from 4.2.
- **Possession by third.** Share of each team's possession spent in its own, middle and attacking third, from that team's own perspective. A high attacking-third share means possession is played high up the pitch.
- **Timeline.** Possession share per `BIN_MINUTES` bin; a short trailing bin is merged into the previous one (see 5.12).

In [ ]:
live = bft_live.copy()                                   # live frames only (stoppages excluded), built in 5.1
lab = live[live["possession_team"].notna()].copy()       # frames with a possession label
car = live[live["carrier_team"].notna()]                 # frames with an identified ball carrier

# ---- possession time and share, two ways ----
sec_labelled = lab.groupby(lab["possession_team"].astype(int)).size() / FPS
sec_carrier = car.groupby(car["carrier_team"].astype(int)).size() / FPS

possession_summary = pd.DataFrame({
    "possession_sec_labelled": sec_labelled,
    "possession_pct_labelled": sec_labelled / sec_labelled.sum() * 100,
    "possession_sec_carrier": sec_carrier,
    "possession_pct_carrier": sec_carrier / sec_carrier.sum() * 100,
})

# ---- possession events (from 4.2) ----
events_by_team = possession_events.groupby(possession_events["possession_team"].astype(int))["duration_sec"].agg(
    n_events="count", mean_event_sec="mean", median_event_sec="median", longest_event_sec="max"
)
possession_summary = possession_summary.join(events_by_team)
possession_summary.index = [TEAM_NAMES[t] for t in possession_summary.index]
possession_pct_carrier = {t: _num(possession_summary.loc[TEAM_NAMES[t], "possession_pct_carrier"]) for t in (0, 1)}

print(f"live frames with a possession label: {len(lab) / len(live) * 100:.1f}% | with an identified carrier: {len(car) / len(live) * 100:.1f}%")
print("\npossession summary (stoppages excluded):")
print(possession_summary.round(1).to_string())

# ---- where each team has the ball: share of its possession by third of the pitch (own / middle / attacking) ----
zone_frames = lab.dropna(subset=["ball_pitch_x"]).copy()
zone_frames["zone"] = [zone_for_team(x, int(t)) for x, t in zip(zone_frames["ball_pitch_x"], zone_frames["possession_team"])]
possession_by_zone = zone_frames.groupby([zone_frames["possession_team"].astype(int), "zone"]).size().unstack(fill_value=0)
possession_by_zone = possession_by_zone.reindex(columns=ZONE_ORDER, fill_value=0)
possession_by_zone_pct = possession_by_zone.div(possession_by_zone.sum(axis=1), axis=0) * 100
possession_by_zone_pct.index = [TEAM_NAMES[t] for t in possession_by_zone_pct.index]
print("\nshare of each team's possession by pitch third (from that team's own perspective, %):")
print(possession_by_zone_pct.round(1).to_string())

# ---- possession share over time ----
lab["time_bin"] = assign_time_bins(lab["frame_idx"])
possession_timeline = lab.groupby(["time_bin", lab["possession_team"].astype(int)]).size().unstack(fill_value=0)
live_sec_per_bin = possession_timeline.sum(axis=1) / FPS
possession_timeline = possession_timeline.div(possession_timeline.sum(axis=1), axis=0) * 100
_last_bin = possession_timeline.index.max()
possession_timeline.index = [bin_label(b, _last_bin) for b in possession_timeline.index]
possession_timeline.columns = [TEAM_NAMES[t] for t in possession_timeline.columns]
possession_timeline["live_sec"] = live_sec_per_bin.values.round(0)
print(f"\npossession share by {BIN_MINUTES}-minute bin (%), live seconds in the last column:")
print(possession_timeline.round(1).to_string())

## 6. Team profile and headline metrics

### 6.1 Metric catalogue
One row per metric: what it measures, its denominator, which team it describes, and how to read it. **Headline** metrics are the ones to quote; **supporting** metrics explain them. This table is exported with the JSON files so the UI and the LLM read the same definitions.

In [ ]:
METRIC_CATALOGUE = pd.DataFrame([
    # ---- headline -------------------------------------------------------------------------------
    dict(metric_id="press_share_pct", tier="headline", credit="pressing team", unit="%", higher_means="more pressing",
         label="Share of opponent possession under pressure", section="5.4",
         definition="Per possession event: share of frames with the opponent pressing; averaged per pressing team, weighted by possession duration.",
         denominator="frames with an identified carrier, inside the opponent's possession events (stoppages excluded)"),
    dict(metric_id="ppda", tier="headline", credit="pressing team", unit="passes/action", higher_means="less pressing",
         label="PPDA", section="5.5",
         definition="Opponent completed passes outside the defending team's own third / own defensive actions (interceptions, tackles) outside that third.",
         denominator="defensive actions in the same zones"),
    dict(metric_id="press_regain_rate_pct", tier="headline", credit="pressing team", unit="%", higher_means="more effective press",
         label="Press -> regain rate", section="5.6",
         definition=f"Share of pressure events after which the pressing team had the ball within {MAX_REGAIN_SEC:g}s of the event ending.",
         denominator="pressure events"),
    dict(metric_id="disruption_forced_pct", tier="headline", credit="pressing team", unit="%", higher_means="more effective press",
         label="Share of opponent's pressured actions disrupted", section="5.9",
         definition="Opponent actions made under pressure that were forced sideways/backward, intercepted, tackled or lost on a long ball.",
         denominator="opponent actions made under pressure (passes, tackles, long-gap transitions)"),
    dict(metric_id="pressed_turnover_rate_own_third_pct", tier="headline", credit="pressing team", unit="%", higher_means="more effective press",
         label="Turnover rate forced, own third", section="5.9",
         definition="Opponent pressured actions in the pressing team's OWN third that ended in a lost ball.", denominator="opponent pressured actions in that zone"),
    dict(metric_id="pressed_turnover_rate_middle_third_pct", tier="headline", credit="pressing team", unit="%", higher_means="more effective press",
         label="Turnover rate forced, middle third", section="5.9",
         definition="Opponent pressured actions in the middle third that ended in a lost ball.", denominator="opponent pressured actions in that zone"),
    dict(metric_id="pressed_turnover_rate_attacking_third_pct", tier="headline", credit="pressing team", unit="%", higher_means="more effective press",
         label="Turnover rate forced, attacking third", section="5.9",
         definition="Opponent pressured actions in the pressing team's ATTACKING third that ended in a lost ball.", denominator="opponent pressured actions in that zone"),
    dict(metric_id="counter_press_fast_regain_pct", tier="headline", credit="team that lost the ball", unit="%", higher_means="stronger counter-press",
         label="Regain rate after a fast counter-press", section="5.7",
         definition=f"After losing the ball, share of turnovers where the team pressed within {COUNTER_PRESS_WINDOW_SEC:g}s and had the ball back within {COUNTER_PRESS_WINDOW_SEC:g}s of the opponent's takeover.",
         denominator="turnovers where the team pressed within the window"),
    # ---- supporting -----------------------------------------------------------------------------
    dict(metric_id="press_time_pct", tier="supporting", credit="pressing team", unit="%", higher_means="more pressing",
         label="Pressing time as share of out-of-possession time", section="5.1",
         definition="Seconds with is_pressing / live out-of-possession seconds.", denominator="live out-of-possession seconds on frames with an identified ball carrier (stoppages excluded)"),
    dict(metric_id="zone_weighted_press_pct", tier="supporting", credit="pressing team", unit="% of max", higher_means="higher, more advanced pressing",
         label="Zone-weighted pressing intensity", section="5.2",
         definition="Pressing frames weighted own third x1, middle x2, attacking x3, over the maximum possible (all out-of-possession frames pressing in the attacking third).",
         denominator="3 x out-of-possession seconds on frames with an identified ball carrier"),
    dict(metric_id="coordinated_press_pct", tier="supporting", credit="pressing team", unit="%", higher_means="more coordinated pressing",
         label="Coordinated presses", section="5.6",
         definition="Share of pressure events with at least two defenders pressing at once.", denominator="pressure events"),
    dict(metric_id="time_to_press_median_sec", tier="supporting", credit="team that lost the ball", unit="s", higher_means="slower reaction",
         label="Median time to press after losing the ball", section="5.7",
         definition=f"Median seconds from losing the ball to the first is_pressing frame, searched up to {MAX_SEARCH_WINDOW_SEC}s.",
         denominator="turnovers where a press was detected"),
    dict(metric_id="press_follow_rate_pct", tier="supporting", credit="pressing team", unit="%", higher_means="more reactive pressing",
         label="Press follows an opponent reception", section="5.10",
         definition=f"Share of opponent pass receptions followed by pressing within {PRESS_ONSET_WINDOW_SEC:g}s.", denominator="opponent completed-pass receptions"),
    dict(metric_id="defensive_line_m", tier="supporting", credit="defending team", unit="m from own goal", higher_means="higher line",
         label="Defensive line height", section="5.3",
         definition=f"Mean distance from own goal of the last {LINE_HEIGHT_DEFENDERS_N} defenders, all frames, outliers removed.", denominator="frames"),
    dict(metric_id="compactness_pressing_m2", tier="supporting", credit="defending team", unit="m2", higher_means="looser block",
         label="Block area while pressing", section="5.11",
         definition="Depth x width of the outfield block on frames where the team is pressing.", denominator="out-of-possession frames while pressing"),
    dict(metric_id="compactness_baseline_m2", tier="supporting", credit="defending team", unit="m2", higher_means="looser block",
         label="Block area while not pressing", section="5.11",
         definition="Depth x width of the outfield block on out-of-possession frames while not pressing.", denominator="out-of-possession frames not pressing"),
    dict(metric_id="passes_under_pressure_pct", tier="supporting", credit="team in possession", unit="%", higher_means="more pressed",
         label="Own passes made under pressure", section="5.8",
         definition=f"Share of the team's own passes released after sustained opponent pressing (>= {MIN_SUSTAINED_PRESSURE_FRAMES} consecutive frames in the last {PRESSURE_LOOKBACK_SEC:g}s).",
         denominator="the team's own pass attempts"),
    dict(metric_id="pass_success_under_pressure_pct", tier="supporting", credit="team in possession", unit="%", higher_means="better under pressure",
         label="Pass completion under pressure", section="5.8",
         definition="Completed / attempted passes released under pressure.", denominator="the team's own passes under pressure"),
    dict(metric_id="possession_pct_carrier", tier="headline", credit="team in possession", unit="%", higher_means="more possession",
         label="Possession share (carrier identified)", section="5.13",
         definition="Share of live frames with an identified ball carrier in which the team has the ball.",
         denominator="live frames with an identified carrier (stoppages excluded)"),
    dict(metric_id="possession_mean_event_sec", tier="supporting", credit="team in possession", unit="s", higher_means="longer possession spells",
         label="Mean possession spell length", section="5.13",
         definition="Mean duration of the team's possession events (4.2).", denominator="the team's possession events"),
    dict(metric_id="possession_attacking_third_pct", tier="supporting", credit="team in possession", unit="%", higher_means="more possession high up the pitch",
         label="Possession in the attacking third", section="5.13",
         definition="Share of the team's possession frames with the ball in the team's attacking third.",
         denominator="the team's possession frames with a known ball position"),
])
METRIC_CATALOGUE_RECORDS = json.loads(METRIC_CATALOGUE.to_json(orient="records"))

METRIC_CATALOGUE[["metric_id", "tier", "credit", "unit", "higher_means", "label", "section"]]

### 6.2 Per-team pressing profile (UI contract)
The table the Streamlit UI reads as `pressing_profile`. Kept exactly as before. Outcome metrics are re-oriented to the **pressing** team, so `turnover_rate_attacking_third` is what a team's press forced in its own attacking third.

In [ ]:
profile_rows = []
for team_id, opponent_id in [(0, 1), (1, 0)]:
    row = {"team": TEAM_NAMES[team_id]}

    # volume + regain (5.4 / 5.6) -- both indexed by the PRESSING team
    row["pct_opp_possession_pressured"] = (
        final_pressure_by_team.loc[team_id, "duration_weighted_pct_under_pressure"]
        if team_id in final_pressure_by_team.index else np.nan
    )
    row["ppda"] = ppda_by_team.get(team_id, np.nan)
    row["press_event_regain_rate_pct"] = (
        regain_summary.loc[team_id, "regain_rate"] if team_id in regain_summary.index else np.nan
    )

    # outcome/turnover effectiveness by zone (5.9) -- indexed by the team BEING pressured,
    # so this team's press effectiveness reads from the opponent's row. zone_summary's "zone" is
    # relative to the team whose action it is (the opponent here) -- e.g. the opponent's own_third
    # is physically THIS (pressing) team's attacking_third, not its own_third. ZONE_FLIP maps a zone
    # label from this team's own perspective to the physically-equivalent label from the opponent's
    # perspective, so "turnover_rate_own_third" is a turnover forced in the pressing team's own third.
    for zone in ["own_third", "middle_third", "attacking_third"]:
        key = (opponent_id, ZONE_FLIP[zone])
        row[f"turnover_rate_{zone}"] = zone_summary.loc[key, "turnover_rate"] if key in zone_summary.index else np.nan

    # counter-press: this team's own regain rate when THEY pressed fast after losing the ball (5.7)
    fast_row = counter_press_summary[(counter_press_summary["losing_team"] == team_id) & (counter_press_summary["press_response"] == "fast")]
    row["counter_press_fast_regain_rate_pct"] = (
        fast_row["regain_rate_pct"].values[0] if len(fast_row) and fast_row["turnovers"].values[0] > 0 else np.nan
    )

    # baseline: how often does this team's press actually follow an opponent's reception at all (5.10)
    opp_receptions = reception_triggers[reception_triggers["team"] == opponent_id]
    row["press_follow_rate_on_opp_receptions_pct"] = (
        opp_receptions["press_followed"].mean() * 100 if len(opp_receptions) else np.nan
    )

    profile_rows.append(row)

team_pressing_profile = pd.DataFrame(profile_rows).set_index("team").round(1)
print(team_pressing_profile.T)

### 6.3 Headline metrics table
One row per catalogue metric, one column per team. Read the `credit` column before comparing rows: most metrics describe the pressing team, but some describe the team in possession or the team that lost the ball.

In [ ]:
def _lookup(series_or_dict, key):
    """value for a key (or (team, flag) tuple) whose team part may be stored as int or float; None if missing."""
    candidates = [key, (float(key[0]),) + tuple(key[1:])] if isinstance(key, tuple) else [key, float(key)]
    for k in candidates:
        try:
            v = series_or_dict[k]
        except (KeyError, IndexError, TypeError):
            continue
        return _num(v)
    return None

_shape_state = team_shape_pressure.groupby(["team", "is_pressing"])["compactness_area_m2"].mean()
_ttp_median = turnovers.groupby("losing_team")["time_to_press_sec"].median()
_pass_under = pass_table.groupby("passer_team")["under_pressure"].mean() * 100

def _team_values(team_id):
    name, opp = TEAM_NAMES[team_id], 1 - team_id
    prof = team_pressing_profile.loc[name]
    fpb = final_pressure_by_team
    return {
        "press_share_pct": _num(fpb.loc[team_id, "duration_weighted_pct_under_pressure"]) if team_id in fpb.index else None,
        "ppda": _num(ppda_by_team.get(team_id, np.nan)),
        "press_regain_rate_pct": _num(regain_summary.loc[team_id, "regain_rate"]) if team_id in regain_summary.index else None,
        "disruption_forced_pct": _lookup(disruption_rate, opp),
        "pressed_turnover_rate_own_third_pct": _num(prof["turnover_rate_own_third"]),
        "pressed_turnover_rate_middle_third_pct": _num(prof["turnover_rate_middle_third"]),
        "pressed_turnover_rate_attacking_third_pct": _num(prof["turnover_rate_attacking_third"]),
        "counter_press_fast_regain_pct": _num(prof["counter_press_fast_regain_rate_pct"]),
        "press_time_pct": pressing_time_pct[team_id],
        "zone_weighted_press_pct": zone_weighted_pct_of_max[team_id],
        "coordinated_press_pct": _lookup(coordinated_press_share_pct, team_id),
        "time_to_press_median_sec": _lookup(_ttp_median, team_id),
        "press_follow_rate_pct": _num(prof["press_follow_rate_on_opp_receptions_pct"]),
        "defensive_line_m": _num(avg_line_height.get(team_id, np.nan)),
        "compactness_pressing_m2": _lookup(_shape_state, (team_id, True)),
        "compactness_baseline_m2": _lookup(_shape_state, (team_id, False)),
        "passes_under_pressure_pct": _lookup(_pass_under, team_id),
        "pass_success_under_pressure_pct": _num(pass_summary["success_rate"].get((team_id, True), np.nan)),
        "possession_pct_carrier": _num(possession_summary.loc[name, "possession_pct_carrier"]),
        "possession_mean_event_sec": _num(possession_summary.loc[name, "mean_event_sec"]),
        "possession_attacking_third_pct": _num(possession_by_zone_pct.loc[name, "attacking_third"]),
    }

_values = {TEAM_NAMES[t]: _team_values(t) for t in (0, 1)}
headline_metrics = METRIC_CATALOGUE[["metric_id", "label", "tier", "credit", "unit"]].copy()
for _name in _values:
    headline_metrics[_name] = headline_metrics["metric_id"].map(lambda m, n=_name: _values[n].get(m))
headline_metrics[list(_values)] = headline_metrics[list(_values)].astype(float).round(2)
headline_metrics = headline_metrics.set_index("metric_id")

missing = [m for m in METRIC_CATALOGUE["metric_id"] if m not in _values["Barca"]]
assert not missing, f"catalogue metrics without a computed value: {missing}"

headline_metrics

### 6.4 Narrative summary (one paragraph per team, the shape an LLM prompt wants)

In [ ]:
def _f(v, d=1):
    return "n/a" if v is None or pd.isna(v) else f"{v:.{d}f}"

for team_id in [0, 1]:
    name = TEAM_NAMES[team_id]
    r = team_pressing_profile.loc[name]
    print(f"{name}:")
    print(
        f"  Pressed the opponent for {_f(r['pct_opp_possession_pressured'])}% of their possession time "
        f"(PPDA {_f(r['ppda'], 2)}), regaining the ball within {MAX_REGAIN_SEC:g}s of a press event {_f(r['press_event_regain_rate_pct'])}% of the time."
    )
    print(
        f"  Turnover rate forced when pressing: {_f(r['turnover_rate_own_third'])}% (own third) / "
        f"{_f(r['turnover_rate_middle_third'])}% (middle) / {_f(r['turnover_rate_attacking_third'])}% (attacking third)."
    )
    print(
        f"  When they pressed within {COUNTER_PRESS_WINDOW_SEC:g}s of losing the ball themselves, regained it {_f(r['counter_press_fast_regain_rate_pct'])}% of the time."
    )
    print()

## 7. Figures

### 7.1 Average defensive line height (pitch view)

In [ ]:
team0_line_x = avg_line_height[0]
team1_line_x = PITCH_LENGTH - avg_line_height[1]

fig, ax = plt.subplots(figsize=(12, 7.5))

ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, color="black", linewidth=1.2))
ax.axvline(PITCH_LENGTH / 2, color="black", linewidth=0.8)
ax.add_patch(patches.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, fill=False, color="black", linewidth=0.8))

box_depth, box_width = 16.5, 40.3
ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - box_width) / 2), box_depth, box_width, fill=False, color="black", linewidth=0.8))
ax.add_patch(patches.Rectangle((PITCH_LENGTH - box_depth, (PITCH_WIDTH - box_width) / 2), box_depth, box_width, fill=False, color="black", linewidth=0.8))

ax.axvline(team0_line_x, color="#1D9E75", linewidth=3, label=f"Barca line ({team0_line_x:.1f}m from own goal)")
ax.axvline(team1_line_x, color="#534AB7", linewidth=3, label=f"Atletico line ({avg_line_height[1]:.1f}m from own goal)")

ax.set_xlim(-5, PITCH_LENGTH + 5)
ax.set_ylim(-5, PITCH_WIDTH + 5)
ax.set_aspect("equal")
ax.axis("off")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, fontsize=11)
ax.set_title("Average defensive line height (last-3 defenders avg)", fontsize=13, pad=15)

plt.tight_layout()
plt.show()

### 7.2 Pressure heatmaps
Where on the pitch each team presses, normalised to that team's own total so the comparison shows **where** pressure concentrates, not raw volume. Own goal is on the left for Barca and on the right for Atletico. Points outside the pitch (homography noise) are dropped.

In [ ]:
pressure_points = team_pressure_full[team_pressure_full["is_pressing"]].dropna(subset=["press_x", "press_y"]).copy()

pressure_points_clean = pressure_points[
    (pressure_points["press_x"] >= -BOUNDS_TOLERANCE_M) & (pressure_points["press_x"] <= PITCH_LENGTH + BOUNDS_TOLERANCE_M) &
    (pressure_points["press_y"] >= -BOUNDS_TOLERANCE_M) & (pressure_points["press_y"] <= PITCH_WIDTH + BOUNDS_TOLERANCE_M)
].copy()

n_dropped = len(pressure_points) - len(pressure_points_clean)
print(f"dropped {n_dropped} / {len(pressure_points)} pressure points outside pitch bounds (+/- {BOUNDS_TOLERANCE_M}m)")
show(pressure_points_clean.groupby("team").size().rename("pressure_points"), digits=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, team_id, label, cmap in zip(axes, [0, 1], ["Barca", "Atletico"], ["Greens", "Purples"]):
    sub = pressure_points_clean[pressure_points_clean["team"] == team_id]

    ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, color="black", linewidth=1.2))
    ax.axvline(PITCH_LENGTH / 2, color="black", linewidth=0.6)
    ax.add_patch(patches.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, fill=False, color="black", linewidth=0.6))
    box_depth, box_width = 16.5, 40.3
    ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - box_width) / 2), box_depth, box_width, fill=False, color="black", linewidth=0.6))
    ax.add_patch(patches.Rectangle((PITCH_LENGTH - box_depth, (PITCH_WIDTH - box_width) / 2), box_depth, box_width, fill=False, color="black", linewidth=0.6))

    weights = np.full(len(sub), 1.0 / len(sub) * 100)
    hb = ax.hexbin(sub["press_x"], sub["press_y"], C=weights, reduce_C_function=np.sum,
                    gridsize=25, cmap=cmap, mincnt=1, extent=(0, PITCH_LENGTH, 0, PITCH_WIDTH), alpha=0.85)
    cb = plt.colorbar(hb, ax=ax, shrink=0.7)
    cb.set_label("% of team's own pressure", fontsize=9)

    own_goal_side = "left" if team_id == 0 else "right"
    ax.set_title(f"{label} — pressure density, normalized (own goal on {own_goal_side})", fontsize=12)
    ax.set_xlim(-3, PITCH_LENGTH + 3)
    ax.set_ylim(-3, PITCH_WIDTH + 3)
    ax.set_aspect("equal")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Exports

### 8.1 UI export — team-level tables
`ui_export` is the Streamlit UI's contract: every key and column name from v30 is preserved (`shape_by_pressure_state` still uses its legacy `under_team_pressure` column name). Two **additive** keys: `meta.metric_catalogue` and `team.headline_metrics`.

In [ ]:
UI_EXPORT_PATH = getattr(paths, "UI_EXPORT_PATH", "pressure_analysis_export.json")

ui_export = {"meta": {}, "team": {}, "events": {}}

ui_export["meta"] = {
    "team_names": TEAM_NAMES,
    "pitch_length_m": PITCH_LENGTH,
    "pitch_width_m": PITCH_WIDTH,
    "fps": FPS,
    "match_duration_sec": _num(match_duration_sec),
    "stoppage_count": int(len(stoppages)),
    "stoppage_total_sec": _num(stoppages["duration_sec"].sum()),
}

line_height_summary = {
    "last1": _df(line_last1_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]), "team"),
    "last3": _df(line_last3_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]), "team"),
}

shape_summary = _df(
    team_shape.groupby("team")[["depth_m", "width_m", "compactness_area_m2"]].agg(["mean", "median", "std"]).round(2),
    "team",
)
shape_by_pressure_state = _df(
    team_shape_pressure.groupby(["team", "is_pressing"])["compactness_area_m2"].agg(["mean", "median", "count"]).round(2)
    .rename_axis(index={"is_pressing": "under_team_pressure"})   # legacy column name kept for the UI contract
)

ui_export["team"] = {
    # the master per-team table -- volume, PPDA, regain rate, turnover rate by zone, counter-press rate
    "pressing_profile": _df(team_pressing_profile, "team_name"),
    "pressing_intensity_pct": pressing_time_pct,
    "zone_pressure_counts": _df(zone_counts, "team"),
    "zone_pressure_pct": _df(zone_pct.round(1), "team"),
    "zone_weighted_pct_of_max": zone_weighted_pct_of_max,
    "defensive_line_height": line_height_summary,
    "shape_summary": shape_summary,
    "shape_by_pressure_state": shape_by_pressure_state,
    "ppda": {int(k): _num(v) for k, v in ppda_by_team.items()},
    "ppda_detail": {int(k): v for k, v in ppda_detail.items()},
    "final_pressure_by_team": _df(final_pressure_by_team.round(2), "team"),
    "pass_summary": _df(pass_summary.round(1)),
    "outcome_pct": _df(outcome_pct.round(1)),
    "disruption_rate_pct": {int(k): _num(v) for k, v in disruption_rate.round(1).to_dict().items()},
    "zone_outcome_summary": _df(zone_summary),
    "regain_summary": _df(regain_summary, "team"),
    "counter_press_response_summary": _df(counter_press_summary),
    "reception_trigger_rates_pct": _df((reception_triggers.groupby("team")[TRIGGER_COLS + ["press_followed"]].mean() * 100).round(1), "team"),
    "pressing_intensity_timeline": _df(pressing_intensity_timeline.round(1), "bin_label"),
}

# additive keys (new): the metric catalogue and the headline table
ui_export["meta"]["metric_catalogue"] = METRIC_CATALOGUE_RECORDS
ui_export["team"]["headline_metrics"] = json.loads(headline_metrics.reset_index().to_json(orient="records"))
ui_export["team"]["possession_summary"] = _df(possession_summary.round(2), "team_name")
ui_export["team"]["possession_by_zone_pct"] = _df(possession_by_zone_pct.round(1), "team_name")
ui_export["team"]["possession_timeline"] = _df(possession_timeline.round(1), "bin_label")
ui_export["team"]["possession_pct_carrier"] = {int(k): _num(v) for k, v in possession_pct_carrier.items()}
ui_export["meta"]["stoppage_gap_sec"] = STOPPAGE_GAP_SEC

print("team-level sections assembled:", list(ui_export["team"].keys()))

### 8.2 UI export — event tables

In [ ]:
# event-level tables -- moderate row counts (tens to low hundreds), suited to a UI timeline / list view

def _events_with_time(df, frame_cols=("start_frame", "end_frame")):
    out = df.copy()
    for c in frame_cols:
        if c in out.columns:
            out[c.replace("frame", "sec")] = out[c] / FPS
    return out

pressure_events_export = _events_with_time(
    team_pressure_events[
        ["team", "start_frame", "end_frame", "duration_sec", "possession_team",
         "mean_n_pressing", "max_n_pressing", "mean_frac_tight_blocked", "regained_ball"]
    ]
)

possession_events_export = _events_with_time(
    possession_events[
        ["event_id", "possession_team", "pressing_team", "start_frame", "end_frame",
         "duration_sec", "pct_frames_under_pressure"]
    ]
)

stoppages_export = _events_with_time(stoppages[["start_frame", "end_frame", "duration_sec"]])

turnovers_export = turnovers[
    ["losing_team", "loss_frame", "gain_frame", "time_to_press_sec", "press_response", "regained_within_window"]
].copy()
turnovers_export["loss_sec"] = turnovers_export["loss_frame"] / FPS
turnovers_export["gain_sec"] = turnovers_export["gain_frame"] / FPS

# passes trimmed to the fields a UI pass-map/list would actually use -- not every intermediate column
_pass_cols = ["passer_team", "release_frame", "release_x", "outcome", "under_pressure",
              "pass_distance_m", "progression_m", "pass_outcome_detail"]
if "release_y" in pass_table.columns:
    _pass_cols.insert(3, "release_y")
passes_export = pass_table[_pass_cols].copy()
passes_export["release_sec"] = passes_export["release_frame"] / FPS

ui_export["events"] = {
    "pressure_events": _df(pressure_events_export),
    "possession_events": _df(possession_events_export),
    "stoppages": _df(stoppages_export),
    "turnovers": _df(turnovers_export),
    "passes": _df(passes_export),
}

print("event-level tables assembled:", {k: len(v) for k, v in ui_export["events"].items()})

### 8.3 UI export — heatmap grid and write file

In [ ]:
# pressure heatmap, binned into a small grid instead of exporting every raw point (7.2)
x_bins = np.arange(0, PITCH_LENGTH + HEATMAP_BIN_M, HEATMAP_BIN_M)
y_bins = np.arange(0, PITCH_WIDTH + HEATMAP_BIN_M, HEATMAP_BIN_M)

heatmap_by_team = {}
for team_id in [0, 1]:
    sub = pressure_points_clean[pressure_points_clean["team"] == team_id]
    counts, _, _ = np.histogram2d(sub["press_x"], sub["press_y"], bins=[x_bins, y_bins])
    pct_of_team_total = counts / counts.sum() * 100 if counts.sum() > 0 else counts
    cells = []
    for i in range(len(x_bins) - 1):
        for j in range(len(y_bins) - 1):
            if counts[i, j] > 0:
                cells.append({
                    "x_min": float(x_bins[i]), "x_max": float(x_bins[i + 1]),
                    "y_min": float(y_bins[j]), "y_max": float(y_bins[j + 1]),
                    "count": int(counts[i, j]),
                    "pct_of_team_total": round(float(pct_of_team_total[i, j]), 2),
                })
    heatmap_by_team[int(team_id)] = cells

ui_export["team"]["pressure_heatmap_grid"] = heatmap_by_team

with open(UI_EXPORT_PATH, "w") as f:
    json.dump(ui_export, f, indent=2)

print(f"wrote {UI_EXPORT_PATH}")
print("top-level keys:", list(ui_export.keys()))
print("team-level keys:", list(ui_export["team"].keys()))
print("event-level keys:", list(ui_export["events"].keys()))

### 8.4 LLM export — overview
`ui_export` stays untouched. This section builds a **separate** file for a downstream LLM: the same numbers plus a `methodology` block generated from this notebook's own constants, a live-computed `data_quality` block, team names instead of ids, `M:SS` next to every seconds value, and a curated `notable_events` section. It also carries the metric catalogue, so the model knows which team each number describes.

### 8.5 LLM export — methodology and data quality

In [ ]:
# --- meta: methodology + live-computed data-quality caveats ---

def _mmss(sec):
    """seconds -> "M:SS" string, None-safe."""
    if sec is None or (isinstance(sec, float) and np.isnan(sec)):
        return None
    total = int(round(sec))
    m, s = divmod(total, 60)
    return f"{m}:{s:02d}"

def _add_mmss(df, sec_cols):
    """copy of df with a '<col minus _sec>_mmss' column added next to each seconds column given."""
    out = df.copy()
    for c in sec_cols:
        if c in out.columns:
            out[c.replace("_sec", "_mmss")] = out[c].apply(_mmss)
    return out

TEAM_ID_FIELD_NAMES = {"team", "passer_team", "losing_team", "possession_team", "pressing_team", "carrier_team"}

def _is_team_id(x):
    try:
        return int(x) in TEAM_NAMES
    except (TypeError, ValueError):
        return False

def _llm_readable(obj):
    """Deep-walk an already-JSON-safe structure (dicts/lists/scalars) and make every team
    reference name-readable, in one generic pass instead of hand-fixing each table:
      - a dict whose keys are ALL team ids (e.g. {0: ..., 1: ...}) gets its keys replaced by
        team names (this is what fixes ppda / pressing_intensity_pct / heatmap_by_team etc.,
        which are keyed by raw team_id while pressing_profile is keyed by team name)
      - any dict with a known team-id field (e.g. "team": 0, "passer_team": 1) gets a sibling
        "<field>_name" added
    Nothing is deleted -- every original numeric id stays in place for exact joins."""
    if isinstance(obj, dict):
        keys = list(obj.keys())
        if keys and all(_is_team_id(k) for k in keys):
            return {TEAM_NAMES[int(k)]: _llm_readable(v) for k, v in obj.items()}
        new = {k: _llm_readable(v) for k, v in obj.items()}
        for field in TEAM_ID_FIELD_NAMES:
            if field in new and _is_team_id(new[field]):
                new[f"{field}_name"] = TEAM_NAMES[int(new[field])]
        return new
    if isinstance(obj, list):
        return [_llm_readable(x) for x in obj]
    return obj

# computed live (not hardcoded) so these can't go stale on a rerun with different source data
ball_carrier_coverage_pct = round(float(ball_frame_table["carrier_track_id"].notna().mean() * 100), 1)
n_distinct_player_tracks = int(player_frame_table.loc[player_frame_table["role"] == "player", "track_id"].nunique())

llm_export = {"meta": {}, "team": {}, "events": {}}

llm_export["meta"] = {
    "team_names": TEAM_NAMES,
    "pitch_length_m": PITCH_LENGTH,
    "pitch_width_m": PITCH_WIDTH,
    "fps": FPS,
    "match_duration_sec": _num(match_duration_sec),
    "match_duration_mmss": _mmss(match_duration_sec),
    "stoppage_count": int(len(stoppages)),
    "stoppage_total_sec": _num(stoppages["duration_sec"].sum()),
    "methodology": {
        "pressure_definition": (
            f"an off-ball defender counts as pressuring the ball carrier when within "
            f"{PRESSURE_DIST_THRESH_M}m AND at least one of: touch-tight (<{TIGHT_MARK_DIST_M}m), "
            f"closing speed >= {CLOSING_SPEED_THRESH_MPS} m/s over roughly "
            f"{CLOSING_WINDOW_FRAMES / FPS:.1f}s, or stepped out >= {STEP_OUT_MARGIN_M}m ahead of "
            f"their team's own out-of-possession baseline defensive line. Raw proximity alone is "
            f"NOT treated as pressure -- this is meant to exclude a passive deep block that the "
            f"attacker simply walked into."
        ),
        "possession_segmentation": (
            "possession events are segmented from carrier_track_id transitions with keep-last gap "
            f"fill; a no-carrier gap >= {STOPPAGE_GAP_SEC}s is treated as a stoppage rather than a "
            "possession change."
        ),
        "ppda_convention": "PPDA zone is classified relative to the DEFENDING team's own third, not the attacking team's.",
        "defensive_line_convention": (
            f"defensive line height = average distance from own goal of the last-"
            f"{LINE_HEIGHT_DEFENDERS_N} defenders, computed only on out-of-possession frames."
        ),
        "team_shape_convention": (
            f"depth = distance between the back-{BACK_N} and front-{FRONT_N} outfield players "
            f"along the team's own attack direction; width = IQR-trimmed lateral spread "
            f"(multiplier {WIDTH_IQR_MULT}); both restricted to out-of-possession frames."
        ),
        "stoppage_handling": (
            f">= {STOPPAGE_GAP_SEC}s continuous no-carrier gaps are excluded from live-play stats "
            "as stoppages (injury, goal, VAR check, cooling break, etc.)."
        ),
        # the four entries below were added after an LLM report flagged apparent inconsistencies
        # in the exported tables; each was individually checked against the actual computation
        # cells -- most turned out to be different metrics that were never meant to match, not
        # data bugs, but the naming/scope was ambiguous enough to cause a real misread. Documented
        # here rather than "fixed" since the underlying numbers were already correct.
        "event_count_note": (
            "counts named 'n_events' or similar mean different things in different tables and "
            "will not match each other: regain_summary counts individual pressure EVENTS (a team "
            "can have several per possession, or one spanning a possession boundary), while "
            "final_pressure_by_team's n_events counts POSSESSION events instead. A 144-vs-101 "
            "style mismatch between these is expected, not a data error."
        ),
        "zone_outcome_summary_convention": (
            "zone_outcome_summary's 'zone' column is relative to the team the row's actions "
            "belong to (the team being pressured), not the pressing team -- e.g. team=1's "
            "own_third row is about team 1's actions in team 1's own third, and credit for any "
            "disruption/turnover there belongs to team 0's press. pressing_profile's "
            "turnover_rate_<zone> columns are already re-oriented to the PRESSING team's own "
            "zone labels, so the two tables use zone labels from opposite perspectives by design."
        ),
        "pass_summary_vs_outcome_pct_scope": (
            "pass_summary's success_rate is completed/attempted PASSES only. outcome_pct covers a "
            "broader event set (passes + tackles + long-ball-gap events) split into finer outcome "
            "categories (progressed/forced-sideways/forced-backward/lost-intercepted/etc). The two "
            "tables have different denominators and category schemes by design -- e.g. "
            "pass_summary success % and (100 - outcome_pct's lost-category %) will not reconcile, "
            "and that is expected, not an error."
        ),
        "lane_blocking_only_pressure_events": (
            "a team-frame is flagged is_pressing if n_pressing >= "
            f"{N_PRESSING_THRESH} OR frac_tight_blocked >= {FRAC_TIGHT_BLOCKED_THRESH} (passing-"
            "lane blocking alone can qualify, without any proximity presser meeting the "
            "closing-speed/touch-tight/step-out gate from the pressure_definition above). A "
            "pressure event can therefore legitimately show mean_n_pressing and max_n_pressing "
            "near zero throughout if it was driven entirely by lane-blocking rather than "
            "proximity pressure -- check mean_frac_tight_blocked on that event before treating a "
            "low-n_pressing event as a data error."
        ),
        "shape_by_pressure_state_orientation": (
            "shape_by_pressure_state's under_team_pressure column (legacy export name of the internal is_pressing flag) reads as 'this team is being "
            "pressed' but means the OPPOSITE: the team in that row (the pressing side) "
            "is the team APPLYING pressure, so a True row is this team's own shape WHILE THEY "
            "ARE PRESSING, not their shape while under the opponent's press. Confirmed by frame "
            "counts: the True rows match each team's own zone_pressure_counts totals exactly."
        ),
        "disruption_rate_pct_orientation": (
            "disruption_rate_pct uses the SAME orientation as zone_outcome_summary -- team = the "
            "team whose action was disrupted, credit for the disruption belongs to the OPPONENT's "
            "press (see zone_outcome_summary_convention above). It is computed once, directly, "
            "from unrounded event-level outcomes -- reconstructing it by averaging "
            "zone_outcome_summary's already-1-decimal-rounded per-zone rates will drift by a few "
            "tenths of a point from double-rounding; that drift is expected numerical noise, not "
            "an orientation error."
        ),
    },
    "data_quality": {
        "ball_carrier_coverage_pct": ball_carrier_coverage_pct,
        "carrier_coverage_note": (
            "the ball carrier is only identified for this share of frames; frames without a "
            "carrier are excluded from carrier-dependent metrics (pressure, passes, PPDA), which "
            "slightly undercounts fast or contested play."
        ),
        "distinct_player_track_ids": n_distinct_player_tracks,
        "identity_fragmentation_note": (
            "player identities are tracked, not fully re-identified across the whole match, so "
            "this count is higher than the true number of players (camera-angle / occlusion "
            "fragmentation). Player-level stats should be read as per-track-segment, not fully "
            "resolved per-athlete; team-level aggregates are not affected by this."
        ),
    },
}

print("LLM export meta assembled:", list(llm_export["meta"].keys()))

# metric catalogue + naming convention, so the model knows which team each number describes
llm_export["meta"]["metric_catalogue"] = METRIC_CATALOGUE_RECORDS
llm_export["meta"]["methodology"]["metric_naming_convention"] = (
    "is_pressing = the team is applying pressure (credit: pressing team); under_pressure = the team WITH the ball "
    "was being pressed (credit: the opponent's press). Every metric in metric_catalogue states which team it describes "
    "in its 'credit' field. headline_metrics (team section) lists one value per team for every catalogue metric."
)
llm_export["meta"]["methodology"]["out_of_possession_denominator"] = (
    "press_time_pct and zone_weighted_press_pct divide pressing seconds by the team's out-of-possession seconds counted only on "
    "frames where a ball carrier is identified, because pressing can only be detected on those frames. Share-of-time pressing "
    "metrics therefore describe identified-carrier play only."
)
llm_export["meta"]["methodology"]["possession_convention"] = (
    "possession_pct_carrier = share of live frames with an identified ball carrier in which the team has the ball; "
    "possession_pct_labelled uses every live frame with a possession label (gaps filled with the last known team). "
    "possession_by_zone_pct is relative to the team's own attacking direction. Stoppages are excluded from all possession figures."
)
llm_export["meta"]["stoppage_gap_sec"] = STOPPAGE_GAP_SEC
llm_export["meta"]["data_quality"]["live_frames_with_possession_label_pct"] = round(float(bft_live["possession_team"].notna().mean() * 100), 1)
llm_export["meta"]["data_quality"]["live_frames_with_carrier_pct"] = round(float(bft_live["carrier_team"].notna().mean() * 100), 1)

### 8.6 LLM export — event tables

In [ ]:
# --- team-level tables, rekeyed consistently to team names via the generic walker above ---

llm_export["team"] = _llm_readable(ui_export["team"])
print("LLM export team-level keys:", list(llm_export["team"].keys()))

# --- event-level tables: reuse the per-row frames from 8.2, add mm:ss + name-readable teams ---

_sec_cols = {
    "pressure_events": ["start_sec", "end_sec"],
    "possession_events": ["start_sec", "end_sec"],
    "stoppages": ["start_sec", "end_sec"],
    "turnovers": ["loss_sec", "gain_sec", "time_to_press_sec"],
    "passes": ["release_sec"],
}
_source_dfs = {
    "pressure_events": pressure_events_export,
    "possession_events": possession_events_export,
    "stoppages": stoppages_export,
    "turnovers": turnovers_export,
    "passes": passes_export,
}

llm_events = {}
for _name, _df_src in _source_dfs.items():
    _enriched = _add_mmss(_df_src, _sec_cols[_name])
    llm_events[_name] = _llm_readable(_df(_enriched))

llm_export["events"] = llm_events
print("LLM export event-level counts:", {k: len(v) for k, v in llm_events.items()})

### 8.7 LLM export — notable events and write files

In [ ]:
# --- notable_events: a curated set of concrete moments for the LLM to anchor a narrative on ---

def _top(df, col, n, ascending=False):
    return df.sort_values(col, ascending=ascending, na_position="last").head(n)

notable = {}

notable["longest_pressure_events_by_team"] = {
    TEAM_NAMES[tid]: _llm_readable(_df(_add_mmss(
        _top(pressure_events_export[pressure_events_export["team"] == tid], "duration_sec", 3),
        _sec_cols["pressure_events"],
    )))
    for tid in [0, 1]
}

notable["fastest_counter_presses"] = _llm_readable(_df(_add_mmss(
    _top(turnovers_export, "time_to_press_sec", 5, ascending=True), _sec_cols["turnovers"]
)))
notable["slowest_counter_presses"] = _llm_readable(_df(_add_mmss(
    _top(turnovers_export, "time_to_press_sec", 5, ascending=False), _sec_cols["turnovers"]
)))

notable["most_heavily_pressured_possessions"] = _llm_readable(_df(_add_mmss(
    _top(possession_events_export, "pct_frames_under_pressure", 5), _sec_cols["possession_events"]
)))

llm_export["events"]["notable_events"] = notable

# --- two files, both COMPACT (no indent) -- this JSON is sent over the wire as literal text,
# so pretty-printing it (as 8.3 does for the UI file) just burns tokens/cost for
# nothing; indent=2 vs compact is purely cosmetic and inflates size ~50% for zero benefit here.
#
# 1) the SLIM default payload (meta + team aggregates + notable_events, no raw event rows) --
#    this is what a first LLM call should send: the whole tactical picture in a few hundred
#    rows instead of 1,000+.
# 2) the FULL export (the same numbers, compact) -- kept as a reserve/lookup file for a
#    follow-up call that needs specific rows (e.g. "show me every pass under pressure"),
#    not something to paste into a prompt wholesale.

llm_default_payload = {
    "meta": llm_export["meta"],
    "team": llm_export["team"],
    "notable_events": notable,
}

LLM_EXPORT_PATH = getattr(paths, "LLM_EXPORT_PATH", "pressure_analysis_llm_export.json")
LLM_EXPORT_DEFAULT_PATH = getattr(paths, "LLM_EXPORT_DEFAULT_PATH", "pressure_analysis_llm_default.json")

with open(LLM_EXPORT_PATH, "w") as f:
    json.dump(llm_export, f, separators=(",", ":"))
with open(LLM_EXPORT_DEFAULT_PATH, "w") as f:
    json.dump(llm_default_payload, f, separators=(",", ":"))

print(f"wrote {LLM_EXPORT_PATH} (full, compact)")
print(f"wrote {LLM_EXPORT_DEFAULT_PATH} (default payload, compact)")
print("meta keys:", list(llm_export["meta"].keys()))
print("team keys:", list(llm_export["team"].keys()))
print("event keys:", list(llm_export["events"].keys()))
print("notable_events keys:", list(notable.keys()))

### 8.8 Size and cost check
Rough input-token estimate (about 4 characters per token) and per-call cost for the default payload and the full export. Re-check the `PRICING_PER_MTOK` table against the current pricing page before relying on it.

In [ ]:
import os

# pricing as of the last time this cell was checked against
# https://platform.claude.com/docs/en/about-claude/pricing -- $ per million tokens (input, output)
PRICING_PER_MTOK = {
    "Claude Haiku 4.5": (1, 5),
    "Claude Sonnet 5":  (2, 10),
    "Claude Opus 5":    (5, 25),
}
ASSUMED_OUTPUT_TOKENS = 3000  # a detailed multi-section tactical write-up, rough order of magnitude

def _size_report(path, label):
    n_bytes = os.path.getsize(path)
    n_tokens_est = n_bytes / 4  # Anthropic's documented rule of thumb, not an exact tokenizer count
    print(f"{label}: {n_bytes:,} bytes  (~{n_tokens_est:,.0f} input tokens, rough estimate)")
    print(f"{'':>{len(label) + 2}}per-call cost at ~{ASSUMED_OUTPUT_TOKENS:,} output tokens:")
    for model, (in_rate, out_rate) in PRICING_PER_MTOK.items():
        cost = n_tokens_est / 1_000_000 * in_rate + ASSUMED_OUTPUT_TOKENS / 1_000_000 * out_rate
        print(f"    {model:<18} ~${cost:.3f}")
    print()

_size_report(LLM_EXPORT_DEFAULT_PATH, "default payload (meta + team + notable_events)")
_size_report(LLM_EXPORT_PATH, "full export (adds every raw event/pass row)")

## Appendix

### A. QA checkpoints
Data coverage and a few automatic sanity checks. Run after the whole notebook. (Each metric's own table is printed in section 5.)

In [ ]:
print("COVERAGE")
print(f"  stoppages: {len(stoppages)} gaps >= {STOPPAGE_GAP_SEC:g}s, {stoppages['duration_sec'].sum():.1f}s excluded of {match_duration_sec:.1f}s")
print(f"  live frames with a possession label: {bft_live['possession_team'].notna().mean() * 100:.1f}% | with an identified carrier: {bft_live['carrier_team'].notna().mean() * 100:.1f}%")
print(f"  events: {len(possession_events)} possession | {len(team_pressure_events)} pressure | {len(turnovers)} turnovers | "
      f"{len(pass_table)} passes | {len(defensive_actions)} defensive actions")
_close = off_ball_pressure[off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M]
print(f"  close-range defender rows excluded by the pressing gate (passive block): {(~_close['is_presser']).mean() * 100:.1f}%")

print("\nAUTOMATIC SANITY CHECKS")
_checks = []
_pct_cols = ["press_share_pct", "press_regain_rate_pct", "disruption_forced_pct", "press_time_pct",
             "zone_weighted_press_pct", "coordinated_press_pct", "passes_under_pressure_pct", "possession_pct_carrier"]
_bad = headline_metrics.loc[_pct_cols, list(_values)].stack()
_checks.append(("percentage metrics inside 0-100", bool(((_bad.dropna() >= 0) & (_bad.dropna() <= 100)).all())))
_checks.append(("possession shares add up to 100", bool(abs(sum(possession_pct_carrier.values()) - 100) < 0.5)))
_checks.append(("both teams have pressure events", bool(team_pressure_events["team"].nunique() == 2)))
_checks.append(("no possession event has negative duration", bool((possession_events["duration_sec"] >= 0).all())))
_checks.append(("pressing seconds never exceed the out-of-possession seconds they are divided by",
                bool(all(total_pressure_seconds_by_team.get(t, 0) <= out_of_possession_seconds[t] for t in (0, 1)))))
_missing = list(headline_metrics.index[headline_metrics[list(_values)].isna().all(axis=1)])
_checks.append((f"catalogue metrics with no value for either team: {_missing if _missing else 'none'}", not _missing))
_carrier_cov = ball_frame_table["carrier_track_id"].notna().mean() * 100
_checks.append((f"ball-carrier coverage {_carrier_cov:.1f}% (>= 40%)", bool(_carrier_cov >= 40)))
for _name, _ok in _checks:
    print(("PASS  " if _ok else "WARN  ") + _name)

### C. Sensitivity of the pressing share to the step-out condition
Nearly half of the presser rows can qualify only through the "stepped out ahead of the team's own line" condition (3.1). This cell shows how much the team-level pressing share moves if that condition is loosened or removed, against both denominators (all labelled out-of-possession frames vs frames with an identified carrier). It is a frame-level approximation: it skips the event bridging and the minimum event duration of 4.1, so its numbers differ slightly from 5.1.

In [ ]:
def pressing_seconds(step_margin):
    d = off_ball_pressure
    close = d["dist_to_carrier"] < PRESSURE_DIST_THRESH_M
    tight = d["dist_to_carrier"] < TIGHT_MARK_DIST_M
    closing = d["closing_speed_mps"].fillna(0) >= CLOSING_SPEED_THRESH_MPS
    step = (d["dist_from_own_goal"] >= d["team_line_avg"] + step_margin) if step_margin is not None else False
    pres = close & (tight | closing | step)
    n = pres.groupby([d["frame_idx"], d["team"]]).sum().rename("n_pressing").reset_index()
    tf = team_frames[["frame_idx", "team", "frac_tight_blocked", "in_own_box"]].merge(n, on=["frame_idx", "team"], how="left")
    tf["n_pressing"] = tf["n_pressing"].fillna(0)
    flag = ((tf["n_pressing"] >= N_PRESSING_THRESH) | (tf["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)) & ~tf["in_own_box"]
    return flag.groupby(tf["team"]).sum() / FPS

rows = []
for label, margin in [(f"current (step-out {STEP_OUT_MARGIN_M:g} m)", STEP_OUT_MARGIN_M), ("step-out 8 m", 8.0), ("no step-out", None)]:
    ps = pressing_seconds(margin)
    rows.append({
        "variant": label,
        **{f"{TEAM_NAMES[t]} % of labelled oop": round(float(ps.get(t, 0) / out_of_possession_seconds_labelled[t] * 100), 1) for t in (0, 1)},
        **{f"{TEAM_NAMES[t]} % of oop with carrier": round(float(ps.get(t, 0) / out_of_possession_seconds[t] * 100), 1) for t in (0, 1)},
    })
print(pd.DataFrame(rows).set_index("variant").to_string())

_lane_only = (team_frames["is_pressing"] & (team_frames["n_pressing"] < N_PRESSING_THRESH)).groupby(team_frames["team"]).sum() \
    / team_frames.groupby("team")["is_pressing"].sum()
print("\nshare of pressing frames flagged by passing-lane blocking alone (fewer than N_PRESSING_THRESH proximity pressers):")
show(_lane_only.rename("lane_only_share") * 100)

### B. Diagnostic — corner-zone pressure cluster
Checks whether the dense hex near the corner for team 0 in the heatmap is one sustained passage of play or genuinely distributed events. Used to decide whether the `BOX_DEPTH_M` own-box exclusion needs changing.

In [ ]:
## Diagnostic: what's driving the concentrated corner-zone pressure cluster for team 0
## Checks whether the dark hex near the corner is one sustained passage of play or genuinely
## distributed events -- needed before deciding whether to change the in_own_box exclusion.

CORNER_ZONE_X_MAX = 30  # just beyond the 16.5m box exclusion, catches the corner/wide-defending band

corner_zone_events = team_pressure_events[
    (team_pressure_events["team"] == 0)
].copy()

# re-attach zone info by checking the ball position at event midpoint
corner_zone_events["mid_frame"] = ((corner_zone_events["start_frame"] + corner_zone_events["end_frame"]) / 2).astype(int)
ball_x_lookup = ball_frame_table.set_index("frame_idx")["ball_pitch_x"]
corner_zone_events["mid_press_x"] = corner_zone_events["mid_frame"].map(ball_x_lookup)

corner_zone_events = corner_zone_events[
    (corner_zone_events["mid_press_x"] > BOX_DEPTH_M) & (corner_zone_events["mid_press_x"] <= CORNER_ZONE_X_MAX)
].sort_values("duration_sec", ascending=False)

print(f"{TEAM_NAMES[0]} pressure events with midpoint between {BOX_DEPTH_M}m and {CORNER_ZONE_X_MAX}m from own goal:")
print(f"{len(corner_zone_events)} events, {corner_zone_events['duration_sec'].sum():.1f}s total")
print()
print(corner_zone_events[["start_frame", "end_frame", "duration_sec", "mean_n_pressing"]].head(10))